# Notebook 13 — StratLake Native Campaign Execution and Artifact Generation

This is a raw pre-import research/development draft for `christophermoverton/fintech-stratlake-notebook-workflows`.

**Theme:** From campaign evidence review to native campaign execution.

Notebook 13 is the runtime counterpart to Notebook 12. Notebook 12 reviewed whether campaign evidence and promotion-readiness surfaces were present without executing a native campaign. Notebook 13 performs guarded native StratLake campaign execution when explicitly enabled, inventories the artifacts produced, and prepares a handoff back into Notebook 12-style evidence review.

**Target path:**

```text
notebooks/13_stratlake_native_campaign_execution_and_artifact_generation.ipynb
```

**Milestone:**

```text
M16 — Notebook 13 Native Campaign Execution and Artifact Generation Import
```

**Conservative default stance:**

```text
notebook_13_native_campaign_execution_import_ready_runtime_execution_manual
```


## Source-safe and native-command-first posture

`stratlake-trade-engine` remains the source of truth for campaign orchestration, strategy execution, artifact generation, manifests, run registries, metrics, split metrics, reporting, evidence review, governance, and archive checkpointing.

This notebook may:

- discover native StratLake command surfaces,
- inspect command help text,
- validate runtime paths and campaign config availability,
- run native StratLake CLI commands only when execution is explicitly enabled,
- capture command results and artifact inventories,
- summarize produced artifacts without claiming promotion readiness by default,
- write Notebook 13 summary artifacts outside Git.

This notebook must not:

- reimplement campaign orchestration,
- fabricate native artifacts,
- treat notebook-generated configs as native templates,
- commit runtime artifacts,
- claim strategy approval, alpha validation, statistical significance, production readiness, or promotion-grade readiness unless native governance evidence supports those claims.


## Relationship to Notebook 12

Notebook 12 finalized the campaign evidence gap and promotion readiness review workflow. Its final stance was:

```text
notebook_12_cold_smoke_guardrail_matrix_passed_with_no_native_campaign_execution
```

Notebook 12 intentionally did not claim native campaign execution, native dry-run success, complete campaign artifacts, strategy approval, alpha validation, production readiness, statistical significance, CI/runtime equivalence, or promotion-grade readiness.

Notebook 13 addresses the major gaps carried forward by Notebook 12:

| Notebook 12 gap | Notebook 13 response |
|---|---|
| No native campaign execution | Guarded native campaign execution profile |
| No verified native dry-run surface | Help-text discovery before command construction |
| Review artifacts are not native campaign artifacts | Artifact origin classification |
| Generated smoke configs are not native templates | Config source classification |
| Missing native campaign context | Campaign execution, artifact inventory, and handoff summary |
| Promotion evidence incomplete | Optional governance command discovery and caveat recording |


## 1. Install notebook dependencies and app packages

This cell follows the established TestPyPI + PyPI fallback pattern. It is intentionally retained as a notebook cell for Colab/local runtime use. It is source-safe when committed with cleared outputs.


In [ ]:
!pip install -q "pandas-market-calendars>=5.0"
!pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ fintech-market-ingestion
!pip install -q --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ stratlake-trade-engine


## 2. Imports, Colab detection, and display helpers

Helpers below are intentionally defensive. Missing optional packages, missing native commands, or missing artifacts become caveats rather than fabricated evidence.


Uncomment the cell below if you intend to do a full campaign execution run.

In [ ]:
import os

# os.environ["NOTEBOOK13_TEST_PROFILE"] = "campaign_execution_run"
# os.environ["NOTEBOOK13_ALLOW_NATIVE_EXECUTION"] = "true"
# os.environ["RUN_STRATLAKE_INIT"] = "true"

# os.environ["NOTEBOOK13_ALLOW_ARCHIVE_RESTORE"] = "true"
# os.environ["NOTEBOOK13_RESTORE_ARCHIVE_ID"] = "notebook-session-001"
# os.environ["NOTEBOOK13_DRIVE_ARCHIVE_ROOT"] = "/content/drive/MyDrive/stratlake-colab/session_archives"

# os.environ["NOTEBOOK13_CREATE_EXECUTION_CONFIGS"] = "true"
# os.environ["NOTEBOOK13_CAMPAIGN_SYMBOLS"] = "SPY,QQQ,IWM"
# os.environ["NOTEBOOK13_CAMPAIGN_STRATEGIES"] = "momentum_v1"

# os.environ["NOTEBOOK13_MARK_INPUTS_USER_REVIEWED"] = "true"
# os.environ["NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION"] = "true"

In [ ]:
import csv
import importlib
import importlib.metadata as importlib_metadata
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd

try:
    from IPython.display import display, Markdown
except Exception:
    display = None
    Markdown = None

try:
    from google.colab import drive, userdata  # type: ignore
    IN_COLAB = True
except Exception:
    drive = None
    userdata = None
    IN_COLAB = False


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def display_df(df: pd.DataFrame, max_rows: int = 20) -> None:
    if display is not None:
        display(df.head(max_rows))
    else:
        print(df.head(max_rows).to_string(index=False))


def display_markdown(text: str) -> None:
    if display is not None and Markdown is not None:
        display(Markdown(text))
    else:
        print(text)


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def command_available(command: str) -> bool:
    return shutil.which(command) is not None


def tail_text(text: str | None, max_chars: int = 4000) -> str:
    if not text:
        return ""
    return text[-max_chars:]


def safe_json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return value.as_posix()
    if isinstance(value, (datetime,)):
        return value.isoformat()
    return str(value)


def write_json(path: Path, data: Any) -> Path:
    ensure_dir(path.parent)
    path.write_text(json.dumps(data, indent=2, default=safe_json_default), encoding="utf-8")
    return path


def write_dataframe_csv(path: Path, df: pd.DataFrame) -> Path:
    ensure_dir(path.parent)
    df.to_csv(path, index=False)
    return path


## 3. Runtime controls and execution profiles

The committed default profile is intentionally safe:

```text
campaign_execution_preview
```

Real campaign execution requires both:

1. a run-capable profile, and
2. `NOTEBOOK13_ALLOW_NATIVE_EXECUTION=true`.

This double gate prevents accidental campaign runs in source validation, CI, or notebook preview.


In [ ]:
# -----------------------------------------------------------------------------
# Notebook 13 profile selector
# -----------------------------------------------------------------------------
NOTEBOOK13_TEST_PROFILE = os.environ.get(
    "NOTEBOOK13_TEST_PROFILE",
    "campaign_execution_preview",
).strip() or "campaign_execution_preview"

PROFILE_MATRIX: dict[str, dict[str, bool]] = {
    # Source-safe default: inspect workspace and command availability only.
    "campaign_execution_preview": {
        "RUN_FINTECH_SESSION_INIT": False,
        "RUN_STRATLAKE_SESSION_INIT": False,
        "DISCOVER_NATIVE_COMMANDS": True,
        "DISCOVER_CAMPAIGN_CONFIGS": True,
        "RUN_CAMPAIGN_PREFLIGHT": True,
        "RUN_ARCHIVE_RESTORE": False,
        "RUN_NATIVE_CAMPAIGN_EXECUTION": False,
        "RUN_OPTIONAL_REPORT_COMMANDS": False,
        "RUN_OPTIONAL_GOVERNANCE_COMMANDS": False,
        "RUN_ARCHIVE_CHECKPOINT": False,
        "WRITE_NOTEBOOK13_SUMMARY_ARTIFACTS": True,
    },
    # Runtime readiness check: command help, config/source validation, no campaign run.
    "campaign_execution_preflight": {
        "RUN_FINTECH_SESSION_INIT": True,
        "RUN_STRATLAKE_SESSION_INIT": True,
        "DISCOVER_NATIVE_COMMANDS": True,
        "DISCOVER_CAMPAIGN_CONFIGS": True,
        "RUN_CAMPAIGN_PREFLIGHT": True,
        "RUN_ARCHIVE_RESTORE": True,
        "RUN_NATIVE_CAMPAIGN_EXECUTION": False,
        "RUN_OPTIONAL_REPORT_COMMANDS": False,
        "RUN_OPTIONAL_GOVERNANCE_COMMANDS": False,
        "RUN_ARCHIVE_CHECKPOINT": False,
        "WRITE_NOTEBOOK13_SUMMARY_ARTIFACTS": True,
    },
    # Full native campaign execution. Requires NOTEBOOK13_ALLOW_NATIVE_EXECUTION=true.
    "campaign_execution_run": {
        "RUN_FINTECH_SESSION_INIT": True,
        "RUN_STRATLAKE_SESSION_INIT": True,
        "DISCOVER_NATIVE_COMMANDS": True,
        "DISCOVER_CAMPAIGN_CONFIGS": True,
        "RUN_CAMPAIGN_PREFLIGHT": True,
        "RUN_ARCHIVE_RESTORE": True,
        "RUN_NATIVE_CAMPAIGN_EXECUTION": True,
        "RUN_OPTIONAL_REPORT_COMMANDS": True,
        "RUN_OPTIONAL_GOVERNANCE_COMMANDS": True,
        "RUN_ARCHIVE_CHECKPOINT": False,
        "WRITE_NOTEBOOK13_SUMMARY_ARTIFACTS": True,
    },
    # Full native campaign execution plus native archive checkpoint.
    "campaign_execution_run_with_archive_checkpoint": {
        "RUN_FINTECH_SESSION_INIT": True,
        "RUN_STRATLAKE_SESSION_INIT": True,
        "DISCOVER_NATIVE_COMMANDS": True,
        "DISCOVER_CAMPAIGN_CONFIGS": True,
        "RUN_CAMPAIGN_PREFLIGHT": True,
        "RUN_ARCHIVE_RESTORE": True,
        "RUN_NATIVE_CAMPAIGN_EXECUTION": True,
        "RUN_OPTIONAL_REPORT_COMMANDS": True,
        "RUN_OPTIONAL_GOVERNANCE_COMMANDS": True,
        "RUN_ARCHIVE_CHECKPOINT": True,
        "WRITE_NOTEBOOK13_SUMMARY_ARTIFACTS": True,
    },
}

if NOTEBOOK13_TEST_PROFILE not in PROFILE_MATRIX:
    raise ValueError(
        f"Unsupported NOTEBOOK13_TEST_PROFILE={NOTEBOOK13_TEST_PROFILE!r}. "
        f"Supported profiles: {sorted(PROFILE_MATRIX)}"
    )

profile = PROFILE_MATRIX[NOTEBOOK13_TEST_PROFILE].copy()
globals().update(profile)

NOTEBOOK13_ALLOW_NATIVE_EXECUTION = os.environ.get(
    "NOTEBOOK13_ALLOW_NATIVE_EXECUTION",
    "false",
).strip().lower() in {"1", "true", "yes", "y"}

NOTEBOOK13_ALLOW_ARCHIVE_CHECKPOINT = os.environ.get(
    "NOTEBOOK13_ALLOW_ARCHIVE_CHECKPOINT",
    "false",
).strip().lower() in {"1", "true", "yes", "y"}

NOTEBOOK13_ALLOW_ARCHIVE_RESTORE = os.environ.get(
    "NOTEBOOK13_ALLOW_ARCHIVE_RESTORE",
    "false",
).strip().lower() in {"1", "true", "yes", "y"}

# Safety guard: Notebook-generated scaffolding may be used for preflight command-shape
# validation, but it must not be executed as a campaign unless explicitly allowed.
NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION = os.environ.get(
    "NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION",
    "false",
).strip().lower() in {"1", "true", "yes", "y"}


# Shared manual review gate for environment-supplied runtime inputs. This should only
# be set after the campaign config, universe config, and feature root have been
# intentionally selected for the current run. It does not make notebook-generated
# scaffolds execution-ready.
NOTEBOOK13_MARK_INPUTS_USER_REVIEWED = os.environ.get(
    "NOTEBOOK13_MARK_INPUTS_USER_REVIEWED",
    "false",
).strip().lower() in {"1", "true", "yes", "y"}

NOTEBOOK13_MODE = NOTEBOOK13_TEST_PROFILE

profile_status = {
    "notebook": "13",
    "profile": NOTEBOOK13_TEST_PROFILE,
    "mode": NOTEBOOK13_MODE,
    "allow_native_execution": NOTEBOOK13_ALLOW_NATIVE_EXECUTION,
    "allow_archive_restore": NOTEBOOK13_ALLOW_ARCHIVE_RESTORE,
    "allow_archive_checkpoint": NOTEBOOK13_ALLOW_ARCHIVE_CHECKPOINT,
    "allow_notebook_generated_config_execution": NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION,
    "mark_inputs_user_reviewed": NOTEBOOK13_MARK_INPUTS_USER_REVIEWED,
    "effective_flags": profile,
    "default_profile_is_source_safe": NOTEBOOK13_TEST_PROFILE == "campaign_execution_preview",
    "created_at_utc": utc_now_iso(),
}

display_markdown("### Notebook 13 runtime profile")
display_df(pd.DataFrame([profile_status]))


### Manual full execution recipe

Use this only after reviewing the selected campaign config, artifact root, and archive target.

```python
NOTEBOOK13_TEST_PROFILE = "campaign_execution_run"
NOTEBOOK13_ALLOW_NATIVE_EXECUTION = True
NOTEBOOK13_CAMPAIGN_CONFIG = "/path/to/native_or_user_reviewed_campaign.yml"
NOTEBOOK13_UNIVERSE_CONFIG = "/path/to/native_or_user_reviewed_universe.yml"
NOTEBOOK13_FEATURE_INPUT_ROOT = "/path/to/features"
NOTEBOOK13_MARK_INPUTS_USER_REVIEWED = True
NOTEBOOK13_ARTIFACT_ROOT = "/content/stratlake/notebook13_artifacts"

# Optional: restore archived features/configs before preflight/run.
NOTEBOOK13_ALLOW_ARCHIVE_RESTORE = True
NOTEBOOK13_RESTORE_ARCHIVE_ID = "notebook-session-001"
NOTEBOOK13_DRIVE_ARCHIVE_ROOT = "/content/drive/MyDrive/stratlake-colab/session_archives"

# Optional: generate execution-candidate configs from reviewed runtime inputs.
NOTEBOOK13_CREATE_EXECUTION_CONFIGS = True
NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION = True
NOTEBOOK13_CAMPAIGN_SYMBOLS = "SPY,QQQ,IWM"
NOTEBOOK13_CAMPAIGN_STRATEGIES = "momentum_v1"
```

For archive checkpointing:

```python
NOTEBOOK13_TEST_PROFILE = "campaign_execution_run_with_archive_checkpoint"
NOTEBOOK13_ALLOW_NATIVE_EXECUTION = True
NOTEBOOK13_ALLOW_ARCHIVE_CHECKPOINT = True
NOTEBOOK13_DRIVE_ARCHIVE_ROOT = "/content/drive/MyDrive/stratlake-colab/session_archives"
```


For the final preflight immediately before execution, prefer:

```python
NOTEBOOK13_TEST_PROFILE = "campaign_execution_preflight"
NOTEBOOK13_CAMPAIGN_CONFIG = "/path/to/native_or_user_reviewed_campaign.yml"
NOTEBOOK13_UNIVERSE_CONFIG = "/path/to/native_or_user_reviewed_universe.yml"
NOTEBOOK13_FEATURE_INPUT_ROOT = "/path/to/features"
NOTEBOOK13_MARK_INPUTS_USER_REVIEWED = True
```

`NOTEBOOK13_MARK_INPUTS_USER_REVIEWED=true` is a deliberate manual gate. It marks environment-supplied campaign and universe configs as reviewed for execution readiness. Notebook-generated starter scaffolds remain blocked from full native execution.


## 4. Workspace, Google Drive mount, and path setup

Keep `/content` as the active runtime workspace in Colab. Treat Google Drive as archive/session persistence, not as the active application workspace.

Generated runtime artifacts are written outside Git under `artifacts/notebook_13_native_campaign_execution_and_artifact_generation/`.


In [ ]:
WORKSPACE_ROOT = Path(os.environ.get("NOTEBOOK13_WORKSPACE_ROOT", "/content" if IN_COLAB else Path.cwd())).resolve()
REPO_ROOT = Path(os.environ.get("NOTEBOOK13_REPO_ROOT", Path.cwd())).resolve()

FINTECH_ROOT = Path(os.environ.get("NOTEBOOK13_FINTECH_ROOT", WORKSPACE_ROOT / "fintech-marketlake")).resolve()
STRATLAKE_ROOT = Path(os.environ.get("NOTEBOOK13_STRATLAKE_ROOT", WORKSPACE_ROOT / "stratlake")).resolve()

NOTEBOOK13_ARTIFACT_ROOT = Path(
    os.environ.get(
        "NOTEBOOK13_ARTIFACT_ROOT",
        REPO_ROOT / "artifacts" / "notebook_13_native_campaign_execution_and_artifact_generation",
    )
).resolve()

NOTEBOOK13_LOG_ROOT = NOTEBOOK13_ARTIFACT_ROOT / "logs"
NOTEBOOK13_SUMMARY_ROOT = NOTEBOOK13_ARTIFACT_ROOT / "summary"
NOTEBOOK13_INVENTORY_ROOT = NOTEBOOK13_ARTIFACT_ROOT / "inventory"

DRIVE_FOLDER_NAME = os.environ.get("NOTEBOOK13_DRIVE_FOLDER_NAME", "TEST1").strip() or "TEST1"

print("IN_COLAB:", IN_COLAB)
print("Python executable:", sys.executable)
print("Current working directory:", Path.cwd().as_posix())
print("Workspace root:", WORKSPACE_ROOT.as_posix())
print("Repo root:", REPO_ROOT.as_posix())
print("Fintech root:", FINTECH_ROOT.as_posix())
print("StratLake root:", STRATLAKE_ROOT.as_posix())
print("Notebook 13 artifact root:", NOTEBOOK13_ARTIFACT_ROOT.as_posix())

if IN_COLAB and drive is not None:
    drive.mount("/content/drive")
else:
    print("Google Drive mount skipped outside Colab.")

for path in [NOTEBOOK13_ARTIFACT_ROOT, NOTEBOOK13_LOG_ROOT, NOTEBOOK13_SUMMARY_ROOT, NOTEBOOK13_INVENTORY_ROOT]:
    ensure_dir(path)

workspace_status = {
    "workspace_root": WORKSPACE_ROOT,
    "repo_root": REPO_ROOT,
    "fintech_root": FINTECH_ROOT,
    "stratlake_root": STRATLAKE_ROOT,
    "artifact_root": NOTEBOOK13_ARTIFACT_ROOT,
    "artifact_root_outside_git_expected": "artifacts/notebook_13_native_campaign_execution_and_artifact_generation" in NOTEBOOK13_ARTIFACT_ROOT.as_posix(),
    "in_colab": IN_COLAB,
}
display_df(pd.DataFrame([workspace_status]))


## 5. Native StratLake command discovery

Notebook 13 confirms command availability before assuming command flags. It records help text tails and candidate flags so the execution cell can construct commands from advertised surfaces rather than hardcoded assumptions.

Expected/native command candidates include:

```text
stratlake-run-research-campaign
stratlake-run-promotion-governance-report
stratlake-session-archive-bootstrap
stratlake-session-archive-restore-bootstrap
```

Additional optional/reporting surfaces are treated as discoverable candidates, not guaranteed support.


In [ ]:
EXPECTED_STRATLAKE_COMMANDS = [
    # Ground-truth initialization/bootstrap surfaces from StratLake console scripts.
    # Prefer notebook workspace initialization, then session initialization.
    "stratlake-init-notebook",
    "stratlake-init-session",
    # Import/export and archive bootstrap surfaces are discovered but not used as primary workspace init.
    "stratlake-session-export",
    "stratlake-session-import",
    # Primary campaign execution surface found in StratLake campaign workflow materials.
    "stratlake-run-research-campaign",
    # Governance/reporting and session checkpointing surfaces used in later StratLake workflows.
    "stratlake-run-promotion-governance-report",
    "stratlake-session-archive-bootstrap",
    "stratlake-session-archive-restore-bootstrap",
    # Optional candidates. Absence is a caveat, not a failure.
    "stratlake-build-campaign-report",
    "stratlake-build-evidence-review",
    "stratlake-query-catalog",
    "stratlake-catalog-index",
    "stratlake-explore-catalog-evidence",
    "stratlake-export-catalog-lineage",
    "stratlake-validate-config",
    "stratlake-doctor",
    "stratlake-notebook-doctor",
    "stratlake-explain-config",
]

EXPECTED_IMPORT_SURFACES = [
    "src.execution",
    "src.execution.orchestration",
    "src.cli.run_research_campaign",
    "src.cli.run_promotion_governance_report",
    "src.research.reporting.campaign_milestone_report",
    "src.research.governance",
    "src.catalog",
]


def run_command_for_discovery(command: list[str], timeout_seconds: int = 20) -> dict[str, Any]:
    result = {
        "command": " ".join(shlex.quote(part) for part in command),
        "returncode": None,
        "stdout_tail": "",
        "stderr_tail": "",
        "succeeded": False,
        "error": "",
    }
    try:
        completed = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=timeout_seconds,
            check=False,
        )
        result.update(
            {
                "returncode": completed.returncode,
                "stdout_tail": tail_text(completed.stdout),
                "stderr_tail": tail_text(completed.stderr),
                "succeeded": completed.returncode == 0,
            }
        )
    except Exception as exc:
        result["error"] = repr(exc)
    return result


def inspect_command_help(command_name: str) -> dict[str, Any]:
    available = command_available(command_name)
    help_result = None
    help_text = ""
    if available:
        help_result = run_command_for_discovery([command_name, "--help"])
        help_text = "\\n".join(
            [
                help_result.get("stdout_tail", ""),
                help_result.get("stderr_tail", ""),
            ]
        )
    flag_candidates = sorted(set(re.findall(r"(?<!\\w)--[a-zA-Z0-9][a-zA-Z0-9_-]*", help_text)))
    return {
        "command": command_name,
        "available": available,
        "help_checked": bool(available),
        "help_returncode": None if help_result is None else help_result.get("returncode"),
        "help_succeeded": False if help_result is None else help_result.get("succeeded", False),
        "flags_detected": flag_candidates,
        "stdout_tail": "" if help_result is None else help_result.get("stdout_tail", ""),
        "stderr_tail": "" if help_result is None else help_result.get("stderr_tail", ""),
        "error": "" if help_result is None else help_result.get("error", ""),
    }


def inspect_import_surface(module_name: str) -> dict[str, Any]:
    try:
        module = importlib.import_module(module_name)
        return {
            "module": module_name,
            "available": True,
            "module_file": getattr(module, "__file__", ""),
            "error": "",
        }
    except Exception as exc:
        return {
            "module": module_name,
            "available": False,
            "module_file": "",
            "error": repr(exc),
        }


def discover_stratlake_entry_points() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    try:
        eps = importlib_metadata.entry_points()
        console_scripts = eps.select(group="console_scripts") if hasattr(eps, "select") else eps.get("console_scripts", [])
        for ep in console_scripts:
            name = getattr(ep, "name", "")
            value = getattr(ep, "value", "")
            if "stratlake" in name.lower() or "stratlake" in value.lower():
                rows.append(
                    {
                        "entry_point_name": name,
                        "entry_point_value": value,
                        "group": "console_scripts",
                    }
                )
    except Exception as exc:
        rows.append(
            {
                "entry_point_name": "",
                "entry_point_value": "",
                "group": "console_scripts",
                "error": repr(exc),
            }
        )
    return pd.DataFrame(rows)


if DISCOVER_NATIVE_COMMANDS:
    command_discovery_df = pd.DataFrame([inspect_command_help(cmd) for cmd in EXPECTED_STRATLAKE_COMMANDS])
    import_surface_df = pd.DataFrame([inspect_import_surface(module) for module in EXPECTED_IMPORT_SURFACES])
    entry_point_df = discover_stratlake_entry_points()
else:
    command_discovery_df = pd.DataFrame()
    import_surface_df = pd.DataFrame()
    entry_point_df = pd.DataFrame()

display_markdown("### Native command discovery")
display_df(command_discovery_df)

display_markdown("### Import surface discovery")
display_df(import_surface_df)

display_markdown("### StratLake console entry points")
display_df(entry_point_df)


## 6. Initialize or attach Fintech and StratLake sessions

These initialization cells use the ground-truth StratLake console script names for notebook/session setup. The preferred initialization surface is `stratlake-init-notebook`; the fallback is `stratlake-init-session`. Session import/export and archive bootstrap commands are discovered separately, but archive bootstrap remains reserved for post-run checkpointing rather than primary workspace initialization.


In [ ]:

session_init_results: list[dict[str, Any]] = []

FINTECH_INIT_COMMAND = ["fintech-session-init", "--root", str(FINTECH_ROOT)]

# Prefer the actual StratLake notebook/session init console scripts provided by pyproject.
# The archive bootstrap command is intentionally excluded here; it is used later only for
# post-run archive checkpointing after a successful campaign execution.
STRATLAKE_WORKSPACE_INIT_CANDIDATES = [
    {
        "surface": "stratlake_init_notebook",
        "command_name": "stratlake-init-notebook",
        "root_flags": ["--root", "--workspace-root", "--project-root"],
        "extra_env_flags": {
            "--drive-root": "NOTEBOOK13_DRIVE_ROOT",
            "--drive-folder-name": "NOTEBOOK13_DRIVE_FOLDER_NAME",
            "--artifact-root": "NOTEBOOK13_ARTIFACT_ROOT",
        },
    },
    {
        "surface": "stratlake_init_session",
        "command_name": "stratlake-init-session",
        "root_flags": ["--root", "--session-root", "--project-root", "--workspace-root"],
        "extra_env_flags": {
            "--drive-root": "NOTEBOOK13_DRIVE_ROOT",
            "--drive-folder-name": "NOTEBOOK13_DRIVE_FOLDER_NAME",
            "--artifact-root": "NOTEBOOK13_ARTIFACT_ROOT",
        },
    },
]


def _command_help_flags(command_name: str) -> set[str]:
    try:
        row = command_discovery_df.loc[command_discovery_df["command"] == command_name]
        if not row.empty:
            flags_value = row.iloc[0].get("flags_detected", [])
            if isinstance(flags_value, str):
                return set(re.findall(r"--[a-zA-Z0-9][a-zA-Z0-9_-]*", flags_value))
            return set(flags_value or [])
    except Exception:
        pass
    return set()


def _build_workspace_init_command(candidate: dict[str, Any]) -> tuple[list[str], list[str]]:
    command_name = candidate["command_name"]
    flags = _command_help_flags(command_name)
    command = [command_name]
    caveats: list[str] = []

    selected_root_flag = None
    for flag in candidate.get("root_flags", []):
        if not flags or flag in flags:
            selected_root_flag = flag
            break
    if selected_root_flag is None:
        caveats.append("no advertised root flag matched candidate root flags")
    else:
        command.extend([selected_root_flag, str(STRATLAKE_ROOT)])

    for flag, env_name in candidate.get("extra_env_flags", {}).items():
        value = os.environ.get(env_name)
        if value and (not flags or flag in flags):
            command.extend([flag, value])
        elif value and flags and flag not in flags:
            caveats.append(f"{flag} not advertised by help; skipped {env_name}")

    # Some bootstrap surfaces support a non-destructive validation/inspection mode. Add these
    # only when advertised and explicitly requested, so source-safe preflight remains conservative.
    if os.environ.get("NOTEBOOK13_BOOTSTRAP_VALIDATE_AFTER_INIT", "false").strip().lower() in {"1", "true", "yes"}:
        for flag in ["--validate-after-copy", "--inspect-after-copy", "--validate-after-init", "--inspect-after-init"]:
            if flag in flags:
                command.append(flag)

    return command, caveats


if RUN_FINTECH_SESSION_INIT:
    if command_available(FINTECH_INIT_COMMAND[0]):
        result = run_command_for_discovery(FINTECH_INIT_COMMAND, timeout_seconds=60)
        result["surface"] = "fintech_session_init"
        result["command_name"] = FINTECH_INIT_COMMAND[0]
        result["run_requested"] = True
        result["selected"] = True
        session_init_results.append(result)
    else:
        session_init_results.append(
            {
                "surface": "fintech_session_init",
                "command_name": FINTECH_INIT_COMMAND[0],
                "command": " ".join(FINTECH_INIT_COMMAND),
                "run_requested": True,
                "selected": False,
                "succeeded": False,
                "error": "fintech-session-init command unavailable",
            }
        )
else:
    session_init_results.append(
        {
            "surface": "fintech_session_init",
            "command_name": FINTECH_INIT_COMMAND[0],
            "command": " ".join(FINTECH_INIT_COMMAND),
            "run_requested": False,
            "selected": False,
            "succeeded": False,
            "error": "",
        }
    )

if RUN_STRATLAKE_SESSION_INIT:
    selected_init = False
    unavailable_candidates: list[str] = []
    skipped_candidates: list[str] = []
    for candidate in STRATLAKE_WORKSPACE_INIT_CANDIDATES:
        command_name = candidate["command_name"]
        if not command_available(command_name):
            unavailable_candidates.append(command_name)
            session_init_results.append(
                {
                    "surface": candidate["surface"],
                    "command_name": command_name,
                    "command": command_name,
                    "run_requested": True,
                    "selected": False,
                    "succeeded": False,
                    "error": "command unavailable",
                }
            )
            continue

        init_command, command_caveats = _build_workspace_init_command(candidate)
        if len(init_command) == 1:
            skipped_candidates.append(command_name)
            session_init_results.append(
                {
                    "surface": candidate["surface"],
                    "command_name": command_name,
                    "command": command_name,
                    "run_requested": True,
                    "selected": False,
                    "succeeded": False,
                    "error": "available command could not be shaped safely from advertised flags",
                    "caveats": command_caveats,
                }
            )
            continue

        result = run_command_for_discovery(init_command, timeout_seconds=int(os.environ.get("NOTEBOOK13_INIT_TIMEOUT_SECONDS", "120")))
        result["surface"] = candidate["surface"]
        result["command_name"] = command_name
        result["run_requested"] = True
        result["selected"] = True
        result["caveats"] = command_caveats
        result["unavailable_candidates_before_selection"] = unavailable_candidates
        session_init_results.append(result)
        selected_init = True
        break

    if not selected_init:
        session_init_results.append(
            {
                "surface": "stratlake_workspace_init_summary",
                "command_name": " OR ".join(c["command_name"] for c in STRATLAKE_WORKSPACE_INIT_CANDIDATES),
                "command": " OR ".join(c["command_name"] for c in STRATLAKE_WORKSPACE_INIT_CANDIDATES),
                "run_requested": True,
                "selected": False,
                "succeeded": False,
                "error": "no StratLake init-notebook/init-session command candidate available or safely shapeable",
                "unavailable_candidates": unavailable_candidates,
                "skipped_candidates": skipped_candidates,
            }
        )
else:
    session_init_results.append(
        {
            "surface": "stratlake_workspace_init_summary",
            "command_name": " OR ".join(c["command_name"] for c in STRATLAKE_WORKSPACE_INIT_CANDIDATES),
            "command": " OR ".join(c["command_name"] for c in STRATLAKE_WORKSPACE_INIT_CANDIDATES),
            "run_requested": False,
            "selected": False,
            "succeeded": False,
            "error": "",
        }
    )

session_init_df = pd.DataFrame(session_init_results)
display_df(session_init_df)


## 7A. Optional archive restore before preflight

Use StratLake's native `stratlake-session-archive-restore-bootstrap` command to restore archived session material before campaign config discovery. This is intentionally separate from post-run archive checkpointing.

Restore is gated by both the active profile and:

```text
NOTEBOOK13_ALLOW_ARCHIVE_RESTORE=true
```

Typical Colab use:

```python
os.environ["NOTEBOOK13_TEST_PROFILE"] = "campaign_execution_preflight"
os.environ["NOTEBOOK13_ALLOW_ARCHIVE_RESTORE"] = "true"
os.environ["NOTEBOOK13_RESTORE_ARCHIVE_ID"] = "notebook-session-001"
os.environ["NOTEBOOK13_DRIVE_ARCHIVE_ROOT"] = "/content/drive/MyDrive/stratlake-colab/session_archives"
```

Current StratLake restore bootstrap command shape:

```bash
stratlake-session-archive-restore-bootstrap \
  --archive-root /content/drive/MyDrive/stratlake-colab/session_archives/notebook-session-001 \
  --target-root /content/stratlake \
  --validate-before-restore \
  --inspect-before-restore \
  --overwrite-policy overwrite_allowed
```

Notebook 13 restores into the local StratLake workspace, not into Drive. Drive remains the archive/session persistence location.


In [ ]:
NOTEBOOK13_DRIVE_ARCHIVE_ROOT = Path(
    os.environ.get(
        "NOTEBOOK13_DRIVE_ARCHIVE_ROOT",
        "/content/drive/MyDrive/stratlake-colab/session_archives" if IN_COLAB else str(NOTEBOOK13_ARTIFACT_ROOT / "session_archives"),
    )
).expanduser()

archive_restore_requested = bool(globals().get("RUN_ARCHIVE_RESTORE", False))
archive_restore_enabled = bool(archive_restore_requested and NOTEBOOK13_ALLOW_ARCHIVE_RESTORE)
archive_restore_command_name = "stratlake-session-archive-restore-bootstrap"
restore_archive_id = os.environ.get(
    "NOTEBOOK13_RESTORE_ARCHIVE_ID",
    os.environ.get("NOTEBOOK13_ARCHIVE_ID", "notebook-session-001"),
).strip()

# Restore bootstrap uses the current StratLake M43+ restore surface:
#
#   stratlake-session-archive-restore-bootstrap \
#     --archive-root <exact archive pack root> \
#     --target-root <local workspace root> \
#     --validate-before-restore \
#     --inspect-before-restore \
#     --overwrite-policy overwrite_allowed
#
# Treat Google Drive as the archive/session persistence location and restore into
# the local StratLake workspace. Do not pass export/bootstrap flags such as
# --drive-root, --archive-id, --include-features, --include-artifacts, or
# --include-configs to this restore command unless the installed help surface
# clearly advertises a legacy fallback surface.
_exact_restore_archive_root = os.environ.get("NOTEBOOK13_RESTORE_ARCHIVE_ROOT", "").strip()
if _exact_restore_archive_root:
    NOTEBOOK13_RESTORE_ARCHIVE_ROOT = Path(_exact_restore_archive_root).expanduser()
else:
    NOTEBOOK13_RESTORE_ARCHIVE_ROOT = (NOTEBOOK13_DRIVE_ARCHIVE_ROOT / restore_archive_id).expanduser()

NOTEBOOK13_RESTORE_TARGET_ROOT = Path(
    os.environ.get("NOTEBOOK13_RESTORE_TARGET_ROOT", str(STRATLAKE_ROOT))
).expanduser()
NOTEBOOK13_RESTORE_OVERWRITE_POLICY = os.environ.get(
    "NOTEBOOK13_RESTORE_OVERWRITE_POLICY",
    "overwrite_allowed",
).strip() or "overwrite_allowed"


def build_archive_restore_command() -> dict[str, Any]:
    flags = _command_help_flags(archive_restore_command_name) if "_command_help_flags" in globals() else flags_for_command(archive_restore_command_name)
    command = [archive_restore_command_name]
    caveats: list[str] = []
    selected_flags: dict[str, str] = {}

    if not command_available(archive_restore_command_name):
        return {
            "command_name": archive_restore_command_name,
            "command": [],
            "command_string": "",
            "build_succeeded": False,
            "flags_detected": sorted(flags),
            "selected_flags": selected_flags,
            "restore_archive_root": NOTEBOOK13_RESTORE_ARCHIVE_ROOT.as_posix(),
            "restore_target_root": NOTEBOOK13_RESTORE_TARGET_ROOT.as_posix(),
            "caveats": ["archive restore bootstrap command unavailable"],
        }

    def add_supported_flag(candidates: list[str], value: str | Path, label: str, *, required: bool = False) -> bool:
        for flag in candidates:
            if flag in flags:
                command.extend([flag, str(value)])
                selected_flags[label] = flag
                return True
        if required:
            caveats.append(f"no supported {label} flag advertised")
        else:
            caveats.append(f"no supported {label} flag advertised; skipped")
        return False

    def add_boolean_flag(candidates: list[str], label: str, *, requested: bool = True, required: bool = False) -> bool:
        if not requested:
            return False
        for flag in candidates:
            if flag in flags:
                command.append(flag)
                selected_flags[label] = flag
                return True
        if required:
            caveats.append(f"no supported {label} flag advertised")
        else:
            caveats.append(f"{label} requested but no supported flag is advertised")
        return False

    restore_surface = "unknown"

    if "--archive-root" in flags:
        restore_surface = "archive_root_target_root"
        add_supported_flag(["--archive-root"], NOTEBOOK13_RESTORE_ARCHIVE_ROOT, "archive_root", required=True)
        add_supported_flag(["--target-root"], NOTEBOOK13_RESTORE_TARGET_ROOT, "target_root", required=True)

        validate_before_requested = os.environ.get(
            "NOTEBOOK13_VALIDATE_BEFORE_RESTORE",
            "true",
        ).strip().lower() in {"1", "true", "yes", "y"}
        inspect_before_requested = os.environ.get(
            "NOTEBOOK13_INSPECT_BEFORE_RESTORE",
            "true",
        ).strip().lower() in {"1", "true", "yes", "y"}

        add_boolean_flag(
            ["--validate-before-restore"],
            "validate_before_restore",
            requested=validate_before_requested,
            required=False,
        )
        add_boolean_flag(
            ["--inspect-before-restore"],
            "inspect_before_restore",
            requested=inspect_before_requested,
            required=False,
        )

        if "--overwrite-policy" in flags:
            command.extend(["--overwrite-policy", NOTEBOOK13_RESTORE_OVERWRITE_POLICY])
            selected_flags["overwrite_policy"] = "--overwrite-policy"
        else:
            caveats.append("no supported overwrite_policy flag advertised; using CLI default")

        if os.environ.get("NOTEBOOK13_ARCHIVE_RESTORE_DRY_RUN", "false").strip().lower() in {"1", "true", "yes", "y"}:
            add_boolean_flag(["--dry-run"], "dry_run", requested=True, required=False)

        if os.environ.get("NOTEBOOK13_ARCHIVE_RESTORE_JSON", "false").strip().lower() in {"1", "true", "yes", "y"}:
            add_boolean_flag(["--json"], "json", requested=True, required=False)

    else:
        # Conservative fallback for older/non-current restore CLIs that do not advertise
        # --archive-root. The current StratLake restore bootstrap should not enter this path.
        restore_surface = "legacy_drive_root_archive_id"
        add_supported_flag(["--root", "--workspace-root", "--project-root"], NOTEBOOK13_RESTORE_TARGET_ROOT, "destination_root", required=False)
        archive_id_added = add_supported_flag(["--archive-id", "--session-id"], restore_archive_id, "archive_id", required=True)
        drive_root_added = add_supported_flag(["--drive-root", "--source-root", "--archive-parent-root"], NOTEBOOK13_DRIVE_ARCHIVE_ROOT, "drive_archive_root", required=True)
        add_supported_flag(["--copy-policy", "--restore-policy", "--collision-policy"], NOTEBOOK13_RESTORE_OVERWRITE_POLICY, "copy_restore_policy", required=False)

        for flag in ["--include-features", "--include-artifacts", "--include-configs"]:
            if flag in flags:
                command.append(flag)
                selected_flags[flag.lstrip("-").replace("-", "_")] = flag

        for flag in ["--validate-before-restore", "--validate-after-restore", "--validate-after-copy"]:
            if flag in flags and os.environ.get("NOTEBOOK13_VALIDATE_BEFORE_RESTORE", "true").strip().lower() in {"1", "true", "yes", "y"}:
                command.append(flag)
                selected_flags["validate_restore"] = flag
                break

        for flag in ["--inspect-before-restore", "--inspect-after-restore", "--inspect-after-copy"]:
            if flag in flags and os.environ.get("NOTEBOOK13_INSPECT_BEFORE_RESTORE", "true").strip().lower() in {"1", "true", "yes", "y"}:
                command.append(flag)
                selected_flags["inspect_restore"] = flag
                break

        if not archive_id_added or not drive_root_added:
            caveats.append("legacy restore surface is missing required archive-id/drive-root style flags")

    missing_required = any(c.startswith("no supported") and "flag advertised" in c for c in caveats if "archive_root" in c or "target_root" in c or "archive_id" in c or "drive_archive_root" in c)

    return {
        "command_name": archive_restore_command_name,
        "command": command,
        "command_string": " ".join(shlex.quote(part) for part in command),
        "build_succeeded": len(command) > 1 and not missing_required,
        "restore_surface": restore_surface,
        "restore_archive_id": restore_archive_id,
        "drive_archive_root": NOTEBOOK13_DRIVE_ARCHIVE_ROOT.as_posix(),
        "restore_archive_root": NOTEBOOK13_RESTORE_ARCHIVE_ROOT.as_posix(),
        "restore_target_root": NOTEBOOK13_RESTORE_TARGET_ROOT.as_posix(),
        "restore_overwrite_policy": NOTEBOOK13_RESTORE_OVERWRITE_POLICY,
        "flags_detected": sorted(flags),
        "selected_flags": selected_flags,
        "caveats": caveats,
    }


archive_restore_spec = build_archive_restore_command()
archive_restore_result = {
    "archive_restore_requested": archive_restore_requested,
    "archive_restore_enabled": archive_restore_enabled,
    "archive_restore_command": archive_restore_spec.get("command_string", ""),
    "archive_restore_returncode": None,
    "archive_restore_succeeded": False,
    "archive_restore_status": "not_requested" if not archive_restore_requested else "blocked",
    "restore_surface": archive_restore_spec.get("restore_surface", ""),
    "restore_archive_id": restore_archive_id,
    "drive_archive_root": NOTEBOOK13_DRIVE_ARCHIVE_ROOT.as_posix(),
    "restore_archive_root": NOTEBOOK13_RESTORE_ARCHIVE_ROOT.as_posix(),
    "restore_target_root": NOTEBOOK13_RESTORE_TARGET_ROOT.as_posix(),
    "restore_overwrite_policy": NOTEBOOK13_RESTORE_OVERWRITE_POLICY,
    "selected_restore_flags": archive_restore_spec.get("selected_flags", {}),
    "stdout_tail": "",
    "stderr_tail": "",
    "error": "",
    "caveats": list(archive_restore_spec.get("caveats", [])),
    "created_at_utc": utc_now_iso(),
}

archive_restore_blockers: list[str] = []
if archive_restore_requested and not NOTEBOOK13_ALLOW_ARCHIVE_RESTORE:
    archive_restore_blockers.append("NOTEBOOK13_ALLOW_ARCHIVE_RESTORE is not true")
if archive_restore_requested and not archive_restore_spec.get("build_succeeded", False):
    archive_restore_blockers.append("archive restore command could not be built safely")
if archive_restore_requested and not command_available(archive_restore_command_name):
    archive_restore_blockers.append("archive restore bootstrap command unavailable")

restore_surface = archive_restore_spec.get("restore_surface", "")
if archive_restore_requested and restore_surface.startswith("archive_root") and not NOTEBOOK13_RESTORE_ARCHIVE_ROOT.exists():
    archive_restore_blockers.append("restore archive root does not exist")
elif archive_restore_requested and not restore_surface.startswith("archive_root") and not NOTEBOOK13_DRIVE_ARCHIVE_ROOT.exists():
    archive_restore_blockers.append("drive/archive root does not exist")

if archive_restore_requested and not NOTEBOOK13_RESTORE_TARGET_ROOT.exists():
    archive_restore_blockers.append("restore target root does not exist")

archive_restore_result["caveats"] = archive_restore_blockers + archive_restore_result["caveats"]

if archive_restore_enabled and not archive_restore_blockers:
    start = time.perf_counter()
    try:
        completed = subprocess.run(
            archive_restore_spec["command"],
            cwd=str(NOTEBOOK13_RESTORE_TARGET_ROOT if NOTEBOOK13_RESTORE_TARGET_ROOT.exists() else WORKSPACE_ROOT),
            capture_output=True,
            text=True,
            timeout=int(os.environ.get("NOTEBOOK13_RESTORE_TIMEOUT_SECONDS", "1800")),
            check=False,
        )
        archive_restore_result.update(
            {
                "archive_restore_returncode": completed.returncode,
                "archive_restore_succeeded": completed.returncode == 0,
                "archive_restore_status": "succeeded" if completed.returncode == 0 else "failed",
                "archive_restore_runtime_seconds": time.perf_counter() - start,
                "stdout_tail": tail_text(completed.stdout),
                "stderr_tail": tail_text(completed.stderr),
            }
        )
    except Exception as exc:
        archive_restore_result.update({"archive_restore_status": "error", "error": repr(exc), "archive_restore_runtime_seconds": time.perf_counter() - start})
else:
    archive_restore_result["archive_restore_status"] = "blocked" if archive_restore_requested else "not_requested"

archive_restore_summary_path = NOTEBOOK13_SUMMARY_ROOT / "archive_restore_summary.json"
write_json(archive_restore_summary_path, {"archive_restore_spec": archive_restore_spec, "archive_restore_result": archive_restore_result})

display_markdown("### Archive restore before preflight")
display_df(pd.DataFrame([archive_restore_result]))
print("Wrote:", archive_restore_summary_path.as_posix())


## 7. Campaign config selection and source classification

Prefer a native StratLake campaign config/template when available.

A notebook-generated starter config may be created only for preview/preflight scaffolding and must be labeled as:

```text
campaign_config_is_notebook_generated = true
campaign_config_is_native_template = false
```

Notebook 13 does not call notebook-generated scaffolding a native template.


In [ ]:
NATIVE_CAMPAIGN_CONFIG_PATTERNS = [
    "configs/campaign*.yml",
    "configs/campaign*.yaml",
    "configs/campaigns/*.yml",
    "configs/campaigns/*.yaml",
    "examples/campaign*.yml",
    "examples/campaign*.yaml",
    "docs/examples/*campaign*.yml",
    "docs/examples/*campaign*.yaml",
    "**/*campaign*.yml",
    "**/*campaign*.yaml",
]

ENV_CAMPAIGN_CONFIG = (
    os.environ.get("NOTEBOOK13_CAMPAIGN_CONFIG", "").strip()
    or os.environ.get("STRATLAKE_CAMPAIGN_CONFIG", "").strip()
    or os.environ.get("CAMPAIGN_CONFIG", "").strip()
)

# Execution-grade campaign configs should be native StratLake templates or explicitly
# user-reviewed configs. Notebook-generated scaffolds are allowed for preview/preflight
# command-shape validation only.
NOTEBOOK13_CAMPAIGN_CONFIG_REVIEWED = (
    NOTEBOOK13_MARK_INPUTS_USER_REVIEWED
    or os.environ.get(
        "NOTEBOOK13_CAMPAIGN_CONFIG_REVIEWED",
        "false",
    ).strip().lower() in {"1", "true", "yes", "y"}
)

# For preview/preflight, create a minimal command-shape config if no native/user config is found.
# For execution profiles, this remains opt-in only.
CREATE_NOTEBOOK13_STARTER_CONFIG = os.environ.get(
    "NOTEBOOK13_CREATE_STARTER_CONFIG",
    "true" if NOTEBOOK13_TEST_PROFILE in {"campaign_execution_preview", "campaign_execution_preflight"} else "false",
).strip().lower() in {"1", "true", "yes", "y"}

# Optional execution-candidate configs are generated from reviewed runtime inputs. They are
# real runnable candidates for manual experiments, but they are still not native templates.
CREATE_NOTEBOOK13_EXECUTION_CONFIGS = os.environ.get(
    "NOTEBOOK13_CREATE_EXECUTION_CONFIGS",
    "false",
).strip().lower() in {"1", "true", "yes", "y"}

NOTEBOOK13_GENERATED_CONFIG_ROOT = Path(
    os.environ.get("NOTEBOOK13_GENERATED_CONFIG_ROOT", NOTEBOOK13_ARTIFACT_ROOT / "configs")
).expanduser().resolve()


def discover_candidate_campaign_configs(search_roots: list[Path]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    seen: set[str] = set()
    for root in search_roots:
        root = Path(root).expanduser()
        if not root.exists():
            rows.append(
                {
                    "search_root": root,
                    "candidate_path": "",
                    "exists": False,
                    "source_hint": "missing_search_root",
                }
            )
            continue
        for pattern in NATIVE_CAMPAIGN_CONFIG_PATTERNS:
            for path in root.glob(pattern):
                if path.is_file():
                    resolved = path.resolve().as_posix()
                    if resolved in seen:
                        continue
                    seen.add(resolved)
                    rows.append(
                        {
                            "search_root": root.resolve(),
                            "candidate_path": path.resolve(),
                            "exists": True,
                            "source_hint": "native_candidate_by_repo_pattern",
                            "suffix": path.suffix,
                            "size_bytes": path.stat().st_size,
                        }
                    )
    return pd.DataFrame(rows)


def create_notebook13_starter_config(path: Path) -> Path:
    ensure_dir(path.parent)
    native_campaign_root = NOTEBOOK13_ARTIFACT_ROOT / "native_campaign"
    starter = f"""# Notebook 13 generated starter campaign config.
# This is NOT a native StratLake template and is NOT promotion/execution evidence.
# Replace with a native or user-reviewed campaign config before running a real campaign.
campaign_id: notebook13_generated_preflight_campaign
description: Notebook 13 generated preflight config for command-shape validation only.
strategies: []
universe: []
artifact_root: {native_campaign_root.as_posix()}
metadata:
  generated_by: notebook_13
  generated_for: campaign_execution_preflight
  native_template: false
  execution_grade: false
  requires_user_review_before_execution: true
"""
    path.write_text(starter, encoding="utf-8")
    return path


def write_text_file(path: Path, text: str) -> Path:
    ensure_dir(path.parent)
    path.write_text(text, encoding="utf-8")
    return path


candidate_config_df = discover_candidate_campaign_configs([REPO_ROOT, STRATLAKE_ROOT, WORKSPACE_ROOT])

selected_campaign_config_path: Path | None = None
campaign_config_source = "none"
campaign_config_is_native_template = False
campaign_config_is_notebook_generated = False
campaign_config_is_user_reviewed = False

if ENV_CAMPAIGN_CONFIG:
    selected_campaign_config_path = Path(ENV_CAMPAIGN_CONFIG).expanduser().resolve()
    campaign_config_source = "env_NOTEBOOK13_CAMPAIGN_CONFIG"
    campaign_config_is_native_template = False
    campaign_config_is_user_reviewed = NOTEBOOK13_CAMPAIGN_CONFIG_REVIEWED
elif not candidate_config_df.empty and "candidate_path" in candidate_config_df.columns:
    existing_candidates = candidate_config_df[candidate_config_df["exists"] == True].copy()
    if not existing_candidates.empty:
        selected_campaign_config_path = Path(existing_candidates.iloc[0]["candidate_path"]).resolve()
        campaign_config_source = "discovered_native_candidate"
        campaign_config_is_native_template = True
        campaign_config_is_user_reviewed = True

if selected_campaign_config_path is None and CREATE_NOTEBOOK13_EXECUTION_CONFIGS:
    selected_campaign_config_path = NOTEBOOK13_GENERATED_CONFIG_ROOT / "notebook13_generated_execution_campaign.yml"
    write_text_file(
        selected_campaign_config_path,
        "# Placeholder written before runtime input discovery; overwritten after archive restore/path preflight.\n"
    )
    campaign_config_source = "notebook13_generated_execution_candidate_config"
    campaign_config_is_native_template = False
    campaign_config_is_notebook_generated = True
    campaign_config_is_user_reviewed = NOTEBOOK13_CAMPAIGN_CONFIG_REVIEWED
elif selected_campaign_config_path is None and CREATE_NOTEBOOK13_STARTER_CONFIG:
    selected_campaign_config_path = NOTEBOOK13_ARTIFACT_ROOT / "configs" / "notebook13_generated_preflight_campaign.yml"
    create_notebook13_starter_config(selected_campaign_config_path)
    campaign_config_source = "notebook13_generated_preflight_config"
    campaign_config_is_native_template = False
    campaign_config_is_notebook_generated = True
    campaign_config_is_user_reviewed = False

campaign_config_exists = False if selected_campaign_config_path is None else selected_campaign_config_path.exists()
if campaign_config_exists and campaign_config_is_notebook_generated:
    campaign_config_validation_status = "exists_notebook_generated_preflight_only"
elif campaign_config_exists and campaign_config_is_user_reviewed:
    campaign_config_validation_status = "exists_user_reviewed_not_schema_validated"
elif campaign_config_exists:
    campaign_config_validation_status = "exists_not_schema_validated_requires_review"
else:
    campaign_config_validation_status = "not_validated"

campaign_config_execution_ready = bool(
    campaign_config_exists
    and (
        (not campaign_config_is_notebook_generated and (campaign_config_is_native_template or campaign_config_is_user_reviewed))
        or (
            campaign_config_source == "notebook13_generated_execution_candidate_config"
            and campaign_config_is_user_reviewed
            and NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION
        )
    )
)

campaign_config_review_status = (
    "user_reviewed_env_config" if campaign_config_source == "env_NOTEBOOK13_CAMPAIGN_CONFIG" and campaign_config_is_user_reviewed
    else "native_candidate_auto_reviewed" if campaign_config_source == "discovered_native_candidate" and campaign_config_is_user_reviewed
    else "notebook_generated_execution_candidate_reviewed" if campaign_config_source == "notebook13_generated_execution_candidate_config" and campaign_config_is_user_reviewed
    else "notebook_generated_execution_candidate_requires_review_and_allow_flag" if campaign_config_source == "notebook13_generated_execution_candidate_config"
    else "notebook_generated_preflight_only" if campaign_config_is_notebook_generated
    else "requires_NOTEBOOK13_MARK_INPUTS_USER_REVIEWED_or_NOTEBOOK13_CAMPAIGN_CONFIG_REVIEWED"
)

campaign_config_status = {
    "campaign_config_source": campaign_config_source,
    "campaign_config_path": "" if selected_campaign_config_path is None else selected_campaign_config_path.as_posix(),
    "campaign_config_exists": campaign_config_exists,
    "campaign_config_is_native_template": campaign_config_is_native_template,
    "campaign_config_is_notebook_generated": campaign_config_is_notebook_generated,
    "campaign_config_is_user_reviewed": campaign_config_is_user_reviewed,
    "campaign_config_execution_ready": campaign_config_execution_ready,
    "campaign_config_validation_status": campaign_config_validation_status,
    "campaign_config_review_status": campaign_config_review_status,
    "mark_inputs_user_reviewed": NOTEBOOK13_MARK_INPUTS_USER_REVIEWED,
}

display_markdown("### Candidate campaign configs")
display_df(candidate_config_df)

display_markdown("### Selected campaign config")
display_df(pd.DataFrame([campaign_config_status]))


## 8. Campaign preflight

Preflight validates only the runtime surface and required inputs. It does not claim campaign execution.

Preflight checks include command availability, help-text availability, config source, artifact root, workspace roots, and candidate feature/config/universe files.


In [ ]:
def path_exists_status(name: str, path: Path | None) -> dict[str, Any]:
    return {
        "check_name": name,
        "path": "" if path is None else path.as_posix(),
        "exists": False if path is None else path.exists(),
        "is_file": False if path is None else path.is_file(),
        "is_dir": False if path is None else path.is_dir(),
    }


def env_path(name: str) -> Path | None:
    value = os.environ.get(name, "").strip()
    return Path(value).expanduser().resolve() if value else None


def discover_existing_dirs(search_roots: list[Path], patterns: list[str], max_rows: int = 25) -> list[Path]:
    discovered: list[Path] = []
    seen: set[str] = set()
    for root in search_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            for path in root.glob(pattern):
                if path.is_dir():
                    resolved = path.resolve().as_posix()
                    if resolved not in seen:
                        seen.add(resolved)
                        discovered.append(path.resolve())
                if len(discovered) >= max_rows:
                    return discovered
    return discovered


def discover_existing_files(search_roots: list[Path], patterns: list[str], max_rows: int = 25) -> list[Path]:
    discovered: list[Path] = []
    seen: set[str] = set()
    for root in search_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            for path in root.glob(pattern):
                if path.is_file():
                    resolved = path.resolve().as_posix()
                    if resolved not in seen:
                        seen.add(resolved)
                        discovered.append(path.resolve())
                if len(discovered) >= max_rows:
                    return discovered
    return discovered


def create_notebook13_starter_universe_config(path: Path) -> Path:
    ensure_dir(path.parent)
    universe_yaml = """# Notebook 13 generated starter universe config.
# This is NOT native StratLake promotion evidence.
# Replace with a project universe.yml/universe.yaml before real native execution.
universe:
  symbols: []
metadata:
  generated_by: notebook_13
  generated_for: campaign_execution_preflight
  native_template: false
  execution_grade: false
  requires_user_review_before_execution: true
"""
    path.write_text(universe_yaml, encoding="utf-8")
    return path


def infer_symbols_from_feature_root(feature_root: Path | None, max_symbols: int = 25) -> list[str]:
    if feature_root is None or not feature_root.exists():
        return []
    symbols: list[str] = []
    seen: set[str] = set()
    for path in list(feature_root.rglob("*.parquet"))[:500] + list(feature_root.rglob("*.csv"))[:500]:
        candidates = [path.stem, path.parent.name]
        for candidate in candidates:
            token = re.sub(r"[^A-Za-z0-9_.-]", "", candidate).upper()
            if 1 <= len(token) <= 12 and token not in seen and not token.lower().startswith(("part", "data", "features")):
                seen.add(token)
                symbols.append(token)
                break
        if len(symbols) >= max_symbols:
            break
    return symbols


def count_feature_files(feature_root: Path | None) -> int:
    if feature_root is None or not feature_root.exists():
        return 0
    return sum(1 for p in feature_root.rglob("*") if p.is_file() and p.suffix.lower() in {".parquet", ".csv", ".json", ".duckdb"})


def create_notebook13_execution_universe_config(path: Path, symbols: list[str]) -> Path:
    ensure_dir(path.parent)
    symbol_lines = "\n".join(f"    - {symbol}" for symbol in symbols) if symbols else "    []"
    universe_yaml = f"""# Notebook 13 generated execution-candidate universe config.
# This is not a native StratLake template. Use only after explicit user review.
universe:
  symbols:
{symbol_lines}
metadata:
  generated_by: notebook_13
  generated_for: native_campaign_execution_candidate
  native_template: false
  execution_grade_candidate: true
  requires_user_review_before_execution: true
"""
    path.write_text(universe_yaml, encoding="utf-8")
    return path


NOTEBOOK13_NATIVE_STRATEGY_ALIASES = {
    # Notebook-friendly aliases mapped onto the native StratLake strategy registry.
    "momentum": "momentum_v1",
    "mean_reversion_safe": "mean_reversion_v1_safe_2026_q1",
    "buy_and_hold": "buy_and_hold_v1",
    "sma_crossover": "sma_crossover_v1",
    "random": "seeded_random_v1",
}


def load_native_strategy_catalog_names(strategy_config_path: Path | None) -> set[str]:
    """Return top-level strategy names from a native StratLake strategies.yml file.

    This intentionally avoids importing StratLake internals so the notebook can fail
    early and clearly even when the native runtime is only partially initialized.
    """
    if strategy_config_path is None or not strategy_config_path.exists():
        return set()
    names: set[str] = set()
    try:
        for line in strategy_config_path.read_text(encoding="utf-8").splitlines():
            if not line.strip() or line.lstrip().startswith("#"):
                continue
            match = re.match(r"^([A-Za-z0-9_.-]+):\s*(?:#.*)?$", line)
            if match:
                names.add(match.group(1))
    except OSError:
        return set()
    return names


def first_existing_path(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.exists():
            return path
    return None



def yaml_scalar_path(path: Path | None) -> str:
    """Return a YAML-safe scalar for native config path fields."""
    if path is None:
        return ""
    return path.as_posix()


def resolve_notebook13_execution_strategies(
    requested_strategies: list[str],
    strategy_config_path: Path | None,
) -> dict[str, Any]:
    requested = requested_strategies or ["momentum_v1"]
    catalog_names = load_native_strategy_catalog_names(strategy_config_path)
    resolved: list[str] = []
    aliases_applied: dict[str, str] = {}
    unknown: list[str] = []

    for strategy in requested:
        candidate = NOTEBOOK13_NATIVE_STRATEGY_ALIASES.get(strategy, strategy)
        if candidate != strategy:
            aliases_applied[strategy] = candidate
        if catalog_names and candidate not in catalog_names:
            unknown.append(candidate)
        elif candidate not in resolved:
            resolved.append(candidate)

    return {
        "requested": requested,
        "resolved": resolved,
        "aliases_applied": aliases_applied,
        "unknown": unknown,
        "catalog_path": "" if strategy_config_path is None else strategy_config_path.as_posix(),
        "catalog_path_exists": False if strategy_config_path is None else strategy_config_path.exists(),
        "catalog_count": len(catalog_names),
        "catalog_preview": sorted(catalog_names)[:25],
    }


def create_notebook13_execution_campaign_config(
    path: Path,
    universe_path: Path,
    feature_root: Path | None,
    strategies: list[str],
    *,
    alpha_names: list[str] | None = None,
    alpha_catalog_path: Path | None,
    strategy_config_path: Path | None,
    portfolio_config_path: Path | None,
) -> Path:
    """Write a native-compatible StratLake research_campaign config.

    Notebook 13 still labels this as a notebook-generated execution candidate rather
    than a native template. It becomes executable only when the explicit review gates
    are enabled, but the YAML shape itself follows src.config.research_campaign.
    """
    ensure_dir(path.parent)
    native_campaign_root = NOTEBOOK13_ARTIFACT_ROOT / "native_campaign"
    alpha_names = alpha_names or []
    alpha_items = "\n".join(f"      - {alpha}" for alpha in alpha_names) if alpha_names else "      []"
    strategy_items = "\n".join(f"      - {strategy}" for strategy in strategies) if strategies else "      []"
    tickers_path_text = yaml_scalar_path(universe_path)
    alpha_catalog_path_text = yaml_scalar_path(alpha_catalog_path)
    strategy_config_path_text = yaml_scalar_path(strategy_config_path)
    portfolio_config_path_text = yaml_scalar_path(portfolio_config_path)

    campaign_yaml = f"""# Notebook 13 generated execution-candidate research campaign config.
# This is not a native StratLake template. Use only after explicit user review.
# The schema follows src.config.research_campaign.ResearchCampaignConfig.
research_campaign:
  dataset_selection:
    dataset:
    timeframe:
    evaluation_horizon:
    mapping_name:
    tickers_path: {tickers_path_text}

  time_windows:
    start:
    end:
    train_start:
    train_end:
    predict_start:
    predict_end:

  targets:
    alpha_names:
{alpha_items}
    strategy_names:
{strategy_items}
    portfolio_names: []
    alpha_catalog_path: {alpha_catalog_path_text}
    strategy_config_path: {strategy_config_path_text}
    portfolio_config_path: {portfolio_config_path_text}

  reuse_policy:
    enable_checkpoint_reuse: true
    reuse_prior_stages:
      - preflight
      - research
      - comparison
      - candidate_selection
      - portfolio
      - candidate_review
      - review
    force_rerun_stages: []
    invalidate_downstream_after_stages: []

  comparison:
    enabled: false
    from_registry: true
    top_k:
    alpha_view: combined
    alpha_metric: ic_ir
    alpha_sleeve_metric: sharpe_ratio
    strategy_metric: sharpe_ratio

  candidate_selection:
    enabled: false
    artifacts_root: artifacts/alpha
    alpha_name:
    dataset:
    timeframe:
    evaluation_horizon:
    mapping_name:
    metric: ic_ir
    max_candidates:
    eligibility:
      min_mean_ic:
      min_mean_rank_ic:
      min_ic_ir:
      min_rank_ic_ir:
      min_history_length:
      min_coverage:
    redundancy:
      max_pairwise_correlation:
      min_overlap_observations:
    allocation:
      allocation_method: equal_weight
      max_weight_per_candidate:
      min_allocation_candidate_count:
      min_allocation_weight:
      allocation_weight_sum_tolerance: 1.0e-12
      allocation_rounding_decimals: 12
    execution:
      strict_mode: false
      skip_eligibility: false
      skip_redundancy: false
      skip_allocation: false
      enable_review: false
      no_markdown_review: false
      register_run: false
      from_registry: false
    output:
      path: artifacts/candidate_selection
      registry_path:
      review_output_path:

  portfolio:
    enabled: false
    portfolio_name:
    timeframe:
    from_registry: false
    from_candidate_selection: false
    evaluation_path:
    optimizer_method:

  review:
    filters:
      run_types:
        - alpha_evaluation
        - strategy
        - portfolio
      timeframe:
      dataset:
      alpha_name:
      strategy_name:
      portfolio_name:
      top_k_per_type:
    ranking:
      alpha_evaluation_primary_metric: ic_ir
      alpha_evaluation_secondary_metric: mean_ic
      strategy_primary_metric: sharpe_ratio
      strategy_secondary_metric: total_return
      portfolio_primary_metric: sharpe_ratio
      portfolio_secondary_metric: total_return
    output:
      path:
      emit_plots: true

  milestone_reporting:
    enabled: true
    decision_categories: []
    output:
      include_markdown_report: true
      decision_log_render_formats:
        - markdown
        - text
    sections:
      campaign_scope: true
      selections: true
      key_findings: true
      key_metrics: true
      gate_outcomes: true
      risks: true
      next_steps: true
      open_questions: true
      decision_snapshot: true
      related_artifacts: true
    summary:
      include_stage_counts: true
      include_review_outcome: true

  outputs:
    alpha_artifacts_root: artifacts/alpha
    candidate_selection_output_path: artifacts/candidate_selection
    candidate_selection_registry_path:
    campaign_artifacts_root: {native_campaign_root.as_posix()}
    portfolio_artifacts_root: artifacts/portfolios
    comparison_output_path:
    review_output_path:

  scenarios:
    enabled: false
    matrix: []
    include: []

"""
    path.write_text(campaign_yaml, encoding="utf-8")
    return path


def create_notebook13_empty_alpha_catalog(path: Path) -> Path:
    """Create a minimal native-loadable alpha catalog for strategy-only campaigns.

    Native research-campaign preflight checks targets.alpha_catalog_path even when
    alpha_names is empty. This generated catalog is therefore only valid when the
    campaign has no alpha targets. If alpha targets are requested, a real alpha
    catalog remains required.
    """
    ensure_dir(path.parent)
    path.write_text(
        "# Notebook 13 generated empty alpha catalog for strategy-only campaign execution.\n{}\n",
        encoding="utf-8",
    )
    return path


def resolve_strategy_only_alpha_catalog_path(
    candidate_path: Path | None,
    *,
    alpha_names: list[str],
) -> dict[str, Any]:
    if candidate_path is not None and candidate_path.exists():
        return {
            "path": candidate_path,
            "source": "existing_native_or_env_alpha_catalog",
            "generated": False,
            "exists": True,
            "blocked": False,
            "blocker": "",
        }
    if alpha_names:
        return {
            "path": candidate_path,
            "source": "missing_required_alpha_catalog",
            "generated": False,
            "exists": False,
            "blocked": True,
            "blocker": (
                "native alpha catalog path unavailable while alpha targets are requested: "
                + ("" if candidate_path is None else candidate_path.as_posix())
            ),
        }
    generated_path = NOTEBOOK13_GENERATED_CONFIG_ROOT / "notebook13_generated_empty_alpha_catalog.yml"
    create_notebook13_empty_alpha_catalog(generated_path)
    return {
        "path": generated_path,
        "source": "notebook13_generated_empty_alpha_catalog_for_strategy_only_campaign",
        "generated": True,
        "exists": generated_path.exists(),
        "blocked": False,
        "blocker": "",
    }


primary_campaign_command = "stratlake-run-research-campaign"

ENV_FEATURE_INPUT_ROOT = env_path("NOTEBOOK13_FEATURE_INPUT_ROOT") or env_path("STRATLAKE_FEATURE_INPUT_ROOT")
ENV_UNIVERSE_CONFIG = env_path("NOTEBOOK13_UNIVERSE_CONFIG") or env_path("STRATLAKE_UNIVERSE_CONFIG")
NOTEBOOK13_CAMPAIGN_ID = os.environ.get("NOTEBOOK13_CAMPAIGN_ID", "notebook13_native_campaign_execution").strip() or "notebook13_native_campaign_execution"
NOTEBOOK13_CAMPAIGN_SYMBOLS = [s.strip().upper() for s in os.environ.get("NOTEBOOK13_CAMPAIGN_SYMBOLS", "").split(",") if s.strip()]
NOTEBOOK13_CAMPAIGN_ALPHA_NAMES = [
    s.strip()
    for s in os.environ.get("NOTEBOOK13_CAMPAIGN_ALPHAS", "").split(",")
    if s.strip()
]
NOTEBOOK13_REQUESTED_CAMPAIGN_STRATEGIES = [
    s.strip()
    for s in os.environ.get("NOTEBOOK13_CAMPAIGN_STRATEGIES", "momentum_v1").split(",")
    if s.strip()
]
NOTEBOOK13_NATIVE_ALPHA_CATALOG_PATH = env_path("NOTEBOOK13_ALPHA_CATALOG_PATH") or first_existing_path(
    [
        STRATLAKE_ROOT / "configs" / "alphas.yml",
        Path("configs/alphas.yml"),
        REPO_ROOT / "configs" / "alphas.yml",
    ]
)
NOTEBOOK13_NATIVE_STRATEGY_CONFIG_PATH = env_path("NOTEBOOK13_STRATEGY_CONFIG_PATH") or first_existing_path(
    [
        STRATLAKE_ROOT / "configs" / "strategies.yml",
        Path("configs/strategies.yml"),
        REPO_ROOT / "configs" / "strategies.yml",
    ]
)
NOTEBOOK13_NATIVE_PORTFOLIO_CONFIG_PATH = env_path("NOTEBOOK13_PORTFOLIO_CONFIG_PATH") or first_existing_path(
    [
        STRATLAKE_ROOT / "configs" / "portfolios.yml",
        Path("configs/portfolios.yml"),
        REPO_ROOT / "configs" / "portfolios.yml",
    ]
)
NOTEBOOK13_ALPHA_CATALOG_RESOLUTION = resolve_strategy_only_alpha_catalog_path(
    NOTEBOOK13_NATIVE_ALPHA_CATALOG_PATH,
    alpha_names=NOTEBOOK13_CAMPAIGN_ALPHA_NAMES,
)
NOTEBOOK13_EFFECTIVE_ALPHA_CATALOG_PATH = NOTEBOOK13_ALPHA_CATALOG_RESOLUTION["path"]
NOTEBOOK13_STRATEGY_RESOLUTION = resolve_notebook13_execution_strategies(
    NOTEBOOK13_REQUESTED_CAMPAIGN_STRATEGIES,
    NOTEBOOK13_NATIVE_STRATEGY_CONFIG_PATH,
)
NOTEBOOK13_CAMPAIGN_STRATEGIES = list(NOTEBOOK13_STRATEGY_RESOLUTION["resolved"])

NOTEBOOK13_REQUIRE_FEATURE_FILES_FOR_EXECUTION = os.environ.get(
    "NOTEBOOK13_REQUIRE_FEATURE_FILES_FOR_EXECUTION",
    "true",
).strip().lower() in {"1", "true", "yes", "y"}

CREATE_NOTEBOOK13_STARTER_UNIVERSE_CONFIG = os.environ.get(
    "NOTEBOOK13_CREATE_STARTER_UNIVERSE_CONFIG",
    "true" if NOTEBOOK13_TEST_PROFILE in {"campaign_execution_preview", "campaign_execution_preflight"} else "false",
).strip().lower() in {"1", "true", "yes", "y"}

NOTEBOOK13_UNIVERSE_CONFIG_REVIEWED = (
    NOTEBOOK13_MARK_INPUTS_USER_REVIEWED
    or os.environ.get(
        "NOTEBOOK13_UNIVERSE_CONFIG_REVIEWED",
        "false",
    ).strip().lower() in {"1", "true", "yes", "y"}
)

feature_input_candidates = [
    p for p in [
        ENV_FEATURE_INPUT_ROOT,
        STRATLAKE_ROOT / "data" / "features",
        STRATLAKE_ROOT / "features",
        STRATLAKE_ROOT / "artifacts" / "features",
        FINTECH_ROOT / "data" / "features",
        FINTECH_ROOT / "artifacts" / "features",
        REPO_ROOT / "data" / "features",
        WORKSPACE_ROOT / "data" / "features",
    ] if p is not None
]
feature_input_candidates.extend(
    discover_existing_dirs(
        [WORKSPACE_ROOT, REPO_ROOT, STRATLAKE_ROOT, FINTECH_ROOT],
        ["**/data/features", "**/features", "**/*feature*"],
        max_rows=20,
    )
)

universe_config_candidates = [
    p for p in [
        ENV_UNIVERSE_CONFIG,
        STRATLAKE_ROOT / "configs" / "universe.yml",
        STRATLAKE_ROOT / "configs" / "universe.yaml",
        STRATLAKE_ROOT / "universe.yml",
        STRATLAKE_ROOT / "universe.yaml",
        REPO_ROOT / "configs" / "universe.yml",
        REPO_ROOT / "configs" / "universe.yaml",
        WORKSPACE_ROOT / "configs" / "universe.yml",
        WORKSPACE_ROOT / "configs" / "universe.yaml",
    ] if p is not None
]
universe_config_candidates.extend(
    discover_existing_files(
        [WORKSPACE_ROOT, REPO_ROOT, STRATLAKE_ROOT],
        ["**/universe.yml", "**/universe.yaml", "**/*universe*.yml", "**/*universe*.yaml"],
        max_rows=20,
    )
)

# De-duplicate candidates while preserving order.
def dedupe_paths(paths: list[Path]) -> list[Path]:
    out: list[Path] = []
    seen: set[str] = set()
    for path in paths:
        key = path.resolve().as_posix() if path.exists() else path.as_posix()
        if key not in seen:
            seen.add(key)
            out.append(path)
    return out

feature_input_candidates = dedupe_paths(feature_input_candidates)
universe_config_candidates = dedupe_paths(universe_config_candidates)

selected_feature_input_path = next((p for p in feature_input_candidates if p.exists() and p.is_dir()), None)
selected_universe_config_path = next((p for p in universe_config_candidates if p.exists() and p.is_file()), None)
universe_config_source = "none"
universe_config_is_notebook_generated = False
universe_config_is_user_reviewed = False
feature_input_source = "none"
feature_input_is_user_reviewed = False

if selected_feature_input_path is not None:
    if ENV_FEATURE_INPUT_ROOT and selected_feature_input_path == ENV_FEATURE_INPUT_ROOT:
        feature_input_source = "env_NOTEBOOK13_FEATURE_INPUT_ROOT"
        feature_input_is_user_reviewed = NOTEBOOK13_MARK_INPUTS_USER_REVIEWED
    else:
        feature_input_source = "discovered_feature_candidate"
        feature_input_is_user_reviewed = NOTEBOOK13_MARK_INPUTS_USER_REVIEWED

if selected_universe_config_path is not None:
    if ENV_UNIVERSE_CONFIG and selected_universe_config_path == ENV_UNIVERSE_CONFIG:
        universe_config_source = "env_NOTEBOOK13_UNIVERSE_CONFIG"
        universe_config_is_user_reviewed = NOTEBOOK13_UNIVERSE_CONFIG_REVIEWED
    else:
        universe_config_source = "discovered_universe_candidate"
        universe_config_is_user_reviewed = True
elif CREATE_NOTEBOOK13_STARTER_UNIVERSE_CONFIG:
    selected_universe_config_path = NOTEBOOK13_ARTIFACT_ROOT / "configs" / "notebook13_generated_universe.yml"
    create_notebook13_starter_universe_config(selected_universe_config_path)
    universe_config_candidates.insert(0, selected_universe_config_path)
    universe_config_source = "notebook13_generated_preflight_universe"
    universe_config_is_notebook_generated = True
    universe_config_is_user_reviewed = False

feature_file_count = count_feature_files(selected_feature_input_path)
execution_candidate_symbols = NOTEBOOK13_CAMPAIGN_SYMBOLS or infer_symbols_from_feature_root(selected_feature_input_path) or ["SPY", "QQQ", "IWM"]

if CREATE_NOTEBOOK13_EXECUTION_CONFIGS:
    generated_universe_path = NOTEBOOK13_GENERATED_CONFIG_ROOT / "notebook13_generated_execution_universe.yml"
    create_notebook13_execution_universe_config(generated_universe_path, execution_candidate_symbols)
    selected_universe_config_path = generated_universe_path
    universe_config_candidates.insert(0, selected_universe_config_path)
    universe_config_source = "notebook13_generated_execution_candidate_universe"
    universe_config_is_notebook_generated = True
    universe_config_is_user_reviewed = NOTEBOOK13_UNIVERSE_CONFIG_REVIEWED

    generated_campaign_path = NOTEBOOK13_GENERATED_CONFIG_ROOT / "notebook13_generated_execution_campaign.yml"
    create_notebook13_execution_campaign_config(
        generated_campaign_path,
        selected_universe_config_path,
        selected_feature_input_path,
        NOTEBOOK13_CAMPAIGN_STRATEGIES,
        alpha_names=NOTEBOOK13_CAMPAIGN_ALPHA_NAMES,
        alpha_catalog_path=NOTEBOOK13_EFFECTIVE_ALPHA_CATALOG_PATH,
        strategy_config_path=NOTEBOOK13_NATIVE_STRATEGY_CONFIG_PATH,
        portfolio_config_path=NOTEBOOK13_NATIVE_PORTFOLIO_CONFIG_PATH,
    )
    selected_campaign_config_path = generated_campaign_path
    campaign_config_source = "notebook13_generated_execution_candidate_config"
    campaign_config_is_native_template = False
    campaign_config_is_notebook_generated = True
    campaign_config_is_user_reviewed = NOTEBOOK13_CAMPAIGN_CONFIG_REVIEWED
    campaign_config_status.update(
        {
            "campaign_config_source": campaign_config_source,
            "campaign_config_path": selected_campaign_config_path.as_posix(),
            "campaign_config_exists": selected_campaign_config_path.exists(),
            "campaign_config_is_native_template": campaign_config_is_native_template,
            "campaign_config_is_notebook_generated": campaign_config_is_notebook_generated,
            "campaign_config_is_user_reviewed": campaign_config_is_user_reviewed,
            "campaign_config_execution_ready": bool(
                selected_campaign_config_path.exists()
                and campaign_config_is_user_reviewed
                and NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION
            ),
            "campaign_config_validation_status": "exists_notebook_generated_execution_candidate",
            "campaign_config_review_status": (
                "notebook_generated_execution_candidate_reviewed"
                if campaign_config_is_user_reviewed
                else "notebook_generated_execution_candidate_requires_review"
            ),
        }
    )

REQUIRE_FEATURE_INPUT_DIR_FOR_PREFLIGHT = os.environ.get(
    "NOTEBOOK13_REQUIRE_FEATURE_INPUT_DIR_FOR_PREFLIGHT",
    "false",
).strip().lower() in {"1", "true", "yes", "y"}

REQUIRE_UNIVERSE_CONFIG_FOR_PREFLIGHT = os.environ.get(
    "NOTEBOOK13_REQUIRE_UNIVERSE_CONFIG_FOR_PREFLIGHT",
    "false",
).strip().lower() in {"1", "true", "yes", "y"}

path_preflight_rows = [
    path_exists_status("workspace_root", WORKSPACE_ROOT),
    path_exists_status("repo_root", REPO_ROOT),
    path_exists_status("fintech_root", FINTECH_ROOT),
    path_exists_status("stratlake_root", STRATLAKE_ROOT),
    path_exists_status("artifact_root", NOTEBOOK13_ARTIFACT_ROOT),
    path_exists_status("native_alpha_catalog_path", NOTEBOOK13_NATIVE_ALPHA_CATALOG_PATH),
    path_exists_status("effective_alpha_catalog_path", NOTEBOOK13_EFFECTIVE_ALPHA_CATALOG_PATH),
    path_exists_status("native_strategy_config_path", NOTEBOOK13_NATIVE_STRATEGY_CONFIG_PATH),
    path_exists_status("native_portfolio_config_path", NOTEBOOK13_NATIVE_PORTFOLIO_CONFIG_PATH),
    path_exists_status("selected_campaign_config", selected_campaign_config_path),
    path_exists_status("selected_feature_input_path", selected_feature_input_path),
    path_exists_status("selected_universe_config_path", selected_universe_config_path),
]

for i, candidate in enumerate(feature_input_candidates, start=1):
    path_preflight_rows.append(path_exists_status(f"feature_input_candidate_{i}", candidate))

for i, candidate in enumerate(universe_config_candidates, start=1):
    path_preflight_rows.append(path_exists_status(f"universe_config_candidate_{i}", candidate))

path_preflight_df = pd.DataFrame(path_preflight_rows)

primary_command_row = {}
if not command_discovery_df.empty:
    matches = command_discovery_df[command_discovery_df["command"] == primary_campaign_command]
    if not matches.empty:
        primary_command_row = matches.iloc[0].to_dict()

preflight_caveats: list[str] = []
preflight_hard_blockers: list[str] = []

if NOTEBOOK13_STRATEGY_RESOLUTION.get("aliases_applied"):
    preflight_caveats.append(
        "strategy alias normalization applied: "
        + ", ".join(
            f"{src}->{dst}"
            for src, dst in sorted(NOTEBOOK13_STRATEGY_RESOLUTION["aliases_applied"].items())
        )
    )

if not primary_command_row.get("available", False):
    preflight_hard_blockers.append("primary native campaign command unavailable")

if NOTEBOOK13_ALPHA_CATALOG_RESOLUTION.get("blocked"):
    preflight_hard_blockers.append(str(NOTEBOOK13_ALPHA_CATALOG_RESOLUTION.get("blocker", "native alpha catalog path unavailable")))
elif NOTEBOOK13_ALPHA_CATALOG_RESOLUTION.get("generated"):
    preflight_caveats.append(
        "generated an empty alpha catalog for a strategy-only campaign because no native alpha catalog was found"
    )

if NOTEBOOK13_NATIVE_STRATEGY_CONFIG_PATH is None or not NOTEBOOK13_NATIVE_STRATEGY_CONFIG_PATH.exists():
    preflight_hard_blockers.append(
        "native strategy config path unavailable for generated campaign config: "
        + ("" if NOTEBOOK13_NATIVE_STRATEGY_CONFIG_PATH is None else NOTEBOOK13_NATIVE_STRATEGY_CONFIG_PATH.as_posix())
    )

if NOTEBOOK13_STRATEGY_RESOLUTION.get("unknown"):
    preflight_hard_blockers.append(
        "unknown native strategy target(s): " + ", ".join(NOTEBOOK13_STRATEGY_RESOLUTION["unknown"])
    )

if not NOTEBOOK13_CAMPAIGN_STRATEGIES:
    preflight_hard_blockers.append("no native strategy targets resolved for execution-candidate config")

if not campaign_config_status["campaign_config_exists"]:
    preflight_hard_blockers.append("no selected campaign config exists")

if campaign_config_is_notebook_generated:
    preflight_caveats.append("selected config is notebook-generated and preflight-only; replace before real native execution")
elif not campaign_config_status.get("campaign_config_execution_ready", False):
    preflight_caveats.append("selected config exists but is not marked native/user-reviewed for real execution")

if selected_feature_input_path is None:
    message = "no feature input candidate directory found"
    if REQUIRE_FEATURE_INPUT_DIR_FOR_PREFLIGHT:
        preflight_hard_blockers.append(message)
    else:
        preflight_caveats.append(message)

if selected_feature_input_path is not None and not feature_input_is_user_reviewed and RUN_NATIVE_CAMPAIGN_EXECUTION:
    preflight_caveats.append("selected feature input path exists but is not marked reviewed for real execution")

if selected_feature_input_path is not None and feature_file_count == 0:
    message = "selected feature input path exists but contains no recognized feature files"
    if NOTEBOOK13_REQUIRE_FEATURE_FILES_FOR_EXECUTION and RUN_NATIVE_CAMPAIGN_EXECUTION:
        preflight_hard_blockers.append(message)
    else:
        preflight_caveats.append(message)

if selected_universe_config_path is None:
    message = "no universe config candidate found"
    if REQUIRE_UNIVERSE_CONFIG_FOR_PREFLIGHT:
        preflight_hard_blockers.append(message)
    else:
        preflight_caveats.append(message)
elif universe_config_is_notebook_generated:
    preflight_caveats.append("selected universe config is notebook-generated and preflight-only; replace before real native execution")
elif not universe_config_is_user_reviewed:
    preflight_caveats.append("selected universe config exists but is not marked reviewed for real execution")

campaign_command_shape_preflight_succeeded = bool(
    RUN_CAMPAIGN_PREFLIGHT
    and primary_command_row.get("available", False)
    and campaign_config_status["campaign_config_exists"]
)

campaign_runtime_input_preflight_succeeded = bool(
    campaign_command_shape_preflight_succeeded
    and selected_feature_input_path is not None
    and selected_universe_config_path is not None
)

feature_files_ready_for_execution = bool(
    selected_feature_input_path is not None
    and (feature_file_count > 0 or not NOTEBOOK13_REQUIRE_FEATURE_FILES_FOR_EXECUTION)
)

universe_config_execution_ready = bool(
    selected_universe_config_path is not None
    and selected_universe_config_path.exists()
    and universe_config_is_user_reviewed
    and (
        not universe_config_is_notebook_generated
        or (
            universe_config_source == "notebook13_generated_execution_candidate_universe"
            and NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION
        )
    )
)

native_execution_input_ready = bool(
    campaign_runtime_input_preflight_succeeded
    and campaign_config_status.get("campaign_config_execution_ready", False)
    and universe_config_execution_ready
    and feature_input_is_user_reviewed
    and feature_files_ready_for_execution
)

preflight_status = {
    "campaign_preflight_requested": RUN_CAMPAIGN_PREFLIGHT,
    "campaign_preflight_succeeded": RUN_CAMPAIGN_PREFLIGHT and not preflight_hard_blockers,
    "campaign_command_shape_preflight_succeeded": campaign_command_shape_preflight_succeeded,
    "campaign_runtime_input_preflight_succeeded": campaign_runtime_input_preflight_succeeded,
    "native_execution_input_ready": native_execution_input_ready,
    "primary_campaign_command": primary_campaign_command,
    "primary_campaign_command_available": primary_command_row.get("available", False),
    "primary_campaign_command_help_checked": primary_command_row.get("help_checked", False),
    "campaign_config_path": campaign_config_status["campaign_config_path"],
    "campaign_config_source": campaign_config_source,
    "campaign_config_is_native_template": campaign_config_is_native_template,
    "campaign_config_is_notebook_generated": campaign_config_is_notebook_generated,
    "campaign_config_is_user_reviewed": campaign_config_is_user_reviewed,
    "campaign_config_execution_ready": campaign_config_status.get("campaign_config_execution_ready", False),
    "selected_feature_input_path": "" if selected_feature_input_path is None else selected_feature_input_path.as_posix(),
    "feature_file_count": feature_file_count,
    "feature_files_ready_for_execution": feature_files_ready_for_execution,
    "feature_input_source": feature_input_source,
    "feature_input_is_user_reviewed": feature_input_is_user_reviewed,
    "selected_universe_config_path": "" if selected_universe_config_path is None else selected_universe_config_path.as_posix(),
    "universe_config_source": universe_config_source,
    "universe_config_is_notebook_generated": universe_config_is_notebook_generated,
    "universe_config_is_user_reviewed": universe_config_is_user_reviewed,
    "universe_config_execution_ready": universe_config_execution_ready,
    "execution_candidate_symbols": execution_candidate_symbols,
    "requested_execution_candidate_strategies": NOTEBOOK13_REQUESTED_CAMPAIGN_STRATEGIES,
    "execution_candidate_alpha_names": NOTEBOOK13_CAMPAIGN_ALPHA_NAMES,
    "execution_candidate_strategies": NOTEBOOK13_CAMPAIGN_STRATEGIES,
    "strategy_aliases_applied": NOTEBOOK13_STRATEGY_RESOLUTION.get("aliases_applied", {}),
    "unknown_execution_candidate_strategies": NOTEBOOK13_STRATEGY_RESOLUTION.get("unknown", []),
    "native_alpha_catalog_path": "" if NOTEBOOK13_NATIVE_ALPHA_CATALOG_PATH is None else NOTEBOOK13_NATIVE_ALPHA_CATALOG_PATH.as_posix(),
    "native_alpha_catalog_exists": False if NOTEBOOK13_NATIVE_ALPHA_CATALOG_PATH is None else NOTEBOOK13_NATIVE_ALPHA_CATALOG_PATH.exists(),
    "effective_alpha_catalog_path": "" if NOTEBOOK13_EFFECTIVE_ALPHA_CATALOG_PATH is None else NOTEBOOK13_EFFECTIVE_ALPHA_CATALOG_PATH.as_posix(),
    "effective_alpha_catalog_exists": False if NOTEBOOK13_EFFECTIVE_ALPHA_CATALOG_PATH is None else NOTEBOOK13_EFFECTIVE_ALPHA_CATALOG_PATH.exists(),
    "alpha_catalog_source": NOTEBOOK13_ALPHA_CATALOG_RESOLUTION.get("source", ""),
    "alpha_catalog_generated_for_strategy_only_campaign": bool(NOTEBOOK13_ALPHA_CATALOG_RESOLUTION.get("generated", False)),
    "native_strategy_config_path": NOTEBOOK13_STRATEGY_RESOLUTION.get("catalog_path", ""),
    "native_strategy_config_exists": NOTEBOOK13_STRATEGY_RESOLUTION.get("catalog_path_exists", False),
    "native_strategy_catalog_count": NOTEBOOK13_STRATEGY_RESOLUTION.get("catalog_count", 0),
    "native_portfolio_config_path": "" if NOTEBOOK13_NATIVE_PORTFOLIO_CONFIG_PATH is None else NOTEBOOK13_NATIVE_PORTFOLIO_CONFIG_PATH.as_posix(),
    "native_portfolio_config_exists": False if NOTEBOOK13_NATIVE_PORTFOLIO_CONFIG_PATH is None else NOTEBOOK13_NATIVE_PORTFOLIO_CONFIG_PATH.exists(),
    "create_notebook13_execution_configs": CREATE_NOTEBOOK13_EXECUTION_CONFIGS,
    "allow_notebook_generated_config_execution": NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION,
    "mark_inputs_user_reviewed": NOTEBOOK13_MARK_INPUTS_USER_REVIEWED,
    "require_feature_input_dir_for_preflight": REQUIRE_FEATURE_INPUT_DIR_FOR_PREFLIGHT,
    "require_universe_config_for_preflight": REQUIRE_UNIVERSE_CONFIG_FOR_PREFLIGHT,
    "artifact_root": NOTEBOOK13_ARTIFACT_ROOT.as_posix(),
    "preflight_hard_blocker_count": len(preflight_hard_blockers),
    "preflight_hard_blockers": preflight_hard_blockers,
    "preflight_caveat_count": len(preflight_caveats),
    "preflight_caveats": preflight_caveats,
    "created_at_utc": utc_now_iso(),
}

display_markdown("### Path preflight")
display_df(path_preflight_df)

display_markdown("### Campaign preflight status")
display_df(pd.DataFrame([preflight_status]))


## 9. Build native campaign command from discovered help text

This cell constructs a command only from known/native command names and help-detected flags where possible. If a required flag is not advertised, the command builder records a caveat and avoids pretending support exists.

The primary execution surface is:

```text
stratlake-run-research-campaign
```


In [ ]:
def flags_for_command(command_name: str) -> set[str]:
    if command_discovery_df.empty:
        return set()
    matches = command_discovery_df[command_discovery_df["command"] == command_name]
    if matches.empty:
        return set()
    flags = matches.iloc[0].get("flags_detected", [])
    if isinstance(flags, str):
        # A CSV/JSON round trip may stringify list-like values.
        return set(re.findall(r"--[a-zA-Z0-9][a-zA-Z0-9_-]*", flags))
    try:
        return set(flags)
    except Exception:
        return set()


def add_first_supported_flag(command: list[str], flags: set[str], candidates: list[str], value: str | Path) -> tuple[bool, str]:
    for flag in candidates:
        if flag in flags:
            command.extend([flag, str(value)])
            return True, flag
    return False, ""


def build_native_campaign_command() -> dict[str, Any]:
    command_name = primary_campaign_command
    flags = flags_for_command(command_name)
    caveats: list[str] = []

    if not command_available(command_name):
        return {
            "command_name": command_name,
            "command": [],
            "command_string": "",
            "build_succeeded": False,
            "flags_detected": sorted(flags),
            "caveats": ["primary native campaign command unavailable"],
        }

    if selected_campaign_config_path is None or not selected_campaign_config_path.exists():
        return {
            "command_name": command_name,
            "command": [],
            "command_string": "",
            "build_succeeded": False,
            "flags_detected": sorted(flags),
            "caveats": ["campaign config path missing"],
        }

    command = [command_name]

    config_added, config_flag = add_first_supported_flag(
        command,
        flags,
        ["--config", "--campaign-config", "--config-path"],
        selected_campaign_config_path,
    )
    if not config_added:
        caveats.append("no supported config flag detected in help text")
        # Conservative fallback: some CLIs accept config as a positional argument.
        # Keep this explicit in caveats rather than claiming flag support.
        command.append(str(selected_campaign_config_path))

    root_added, root_flag = add_first_supported_flag(
        command,
        flags,
        ["--root", "--workspace-root", "--project-root"],
        STRATLAKE_ROOT,
    )
    if not root_added:
        caveats.append("no root/workspace flag detected; command may rely on current working directory")

    artifact_added, artifact_flag = add_first_supported_flag(
        command,
        flags,
        ["--artifact-root", "--artifacts-root", "--output-root", "--outputs-root", "--run-root"],
        NOTEBOOK13_ARTIFACT_ROOT / "native_campaign",
    )
    if not artifact_added:
        caveats.append("no artifact/output root flag detected; native default output location may be used")

    if selected_universe_config_path is not None and selected_universe_config_path.exists():
        universe_added, universe_flag = add_first_supported_flag(
            command,
            flags,
            ["--universe-config", "--universe", "--universe-path", "--symbols-config"],
            selected_universe_config_path,
        )
        if not universe_added:
            caveats.append("no universe config flag detected; native command may rely on campaign config/default workspace")
    else:
        universe_flag = ""
        caveats.append("universe config path missing")

    if selected_feature_input_path is not None and selected_feature_input_path.exists():
        feature_added, feature_flag = add_first_supported_flag(
            command,
            flags,
            ["--feature-root", "--features-root", "--feature-input-root", "--features-dir", "--feature-dir"],
            selected_feature_input_path,
        )
        if not feature_added:
            caveats.append("no feature input root flag detected; native command may rely on campaign config/default workspace")
    else:
        feature_flag = ""
        caveats.append("feature input root missing")

    campaign_id = os.environ.get("NOTEBOOK13_CAMPAIGN_ID", "notebook13_native_campaign_execution").strip()
    campaign_id_added, campaign_id_flag = add_first_supported_flag(
        command,
        flags,
        ["--campaign-id", "--run-id", "--execution-id"],
        campaign_id,
    )
    if not campaign_id_added:
        caveats.append("no campaign/run id flag detected; native command may generate its own identifier")

    return {
        "command_name": command_name,
        "command": command,
        "command_string": " ".join(shlex.quote(part) for part in command),
        "build_succeeded": True,
        "flags_detected": sorted(flags),
        "selected_config_flag": config_flag,
        "selected_root_flag": root_flag,
        "selected_artifact_flag": artifact_flag,
        "selected_universe_flag": universe_flag,
        "selected_feature_flag": feature_flag,
        "selected_campaign_id_flag": campaign_id_flag,
        "caveats": caveats,
    }


native_campaign_command_spec = build_native_campaign_command()

display_markdown("### Native campaign command specification")
display_df(pd.DataFrame([native_campaign_command_spec]))


## 10. Native campaign execution

Execution occurs only when all gates are satisfied:

- selected profile requests native execution,
- `NOTEBOOK13_ALLOW_NATIVE_EXECUTION=true`,
- preflight succeeds,
- command build succeeds.

The notebook records whether execution actually occurred. No campaign success is claimed when execution is skipped or blocked.


In [ ]:
campaign_execution_requested = bool(RUN_NATIVE_CAMPAIGN_EXECUTION)
campaign_execution_enabled = bool(RUN_NATIVE_CAMPAIGN_EXECUTION and NOTEBOOK13_ALLOW_NATIVE_EXECUTION)
campaign_execution_result: dict[str, Any] = {
    "campaign_execution_requested": campaign_execution_requested,
    "campaign_execution_enabled": campaign_execution_enabled,
    "campaign_execution_command": native_campaign_command_spec.get("command_string", ""),
    "campaign_execution_returncode": None,
    "campaign_execution_succeeded": False,
    "campaign_execution_runtime_seconds": 0.0,
    "campaign_execution_claim_made": False,
    "campaign_execution_status": "not_requested" if not campaign_execution_requested else "blocked",
    "stdout_tail": "",
    "stderr_tail": "",
    "error": "",
    "caveats": [],
    "created_at_utc": utc_now_iso(),
}

execution_blockers: list[str] = []
if campaign_execution_requested and not NOTEBOOK13_ALLOW_NATIVE_EXECUTION:
    execution_blockers.append("NOTEBOOK13_ALLOW_NATIVE_EXECUTION is not true")
if campaign_execution_requested and not preflight_status.get("campaign_preflight_succeeded", False):
    execution_blockers.append("campaign preflight did not succeed")
if campaign_execution_requested and not native_campaign_command_spec.get("build_succeeded", False):
    execution_blockers.append("native campaign command build did not succeed")
if campaign_execution_requested and not preflight_status.get("native_execution_input_ready", False):
    execution_blockers.append(
        "native execution inputs are not ready; provide campaign config, universe config, feature input root, and set NOTEBOOK13_MARK_INPUTS_USER_REVIEWED=true before execution"
    )
if campaign_execution_requested and campaign_config_is_notebook_generated and not NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION:
    execution_blockers.append(
        "selected campaign config is notebook-generated; set NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION=true only after reviewing the generated execution candidate, or provide a native/user-reviewed config"
    )
if campaign_execution_requested and preflight_status.get("universe_config_is_notebook_generated", False) and not NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION:
    execution_blockers.append(
        "selected universe config is notebook-generated; set NOTEBOOK13_ALLOW_NOTEBOOK_GENERATED_CONFIG_EXECUTION=true only after reviewing the generated execution candidate, or provide a project/user-reviewed universe config"
    )
if campaign_execution_requested and preflight_status.get("feature_file_count", 0) == 0 and NOTEBOOK13_REQUIRE_FEATURE_FILES_FOR_EXECUTION:
    execution_blockers.append("selected feature input root contains no recognized feature files")

execution_caveats = list(native_campaign_command_spec.get("caveats", []))
if campaign_execution_requested:
    execution_caveats = execution_blockers + execution_caveats
elif campaign_config_is_notebook_generated:
    execution_caveats.append("selected campaign config is notebook-generated; execution was not requested")

campaign_execution_result["caveats"] = execution_caveats

if campaign_execution_enabled and not execution_blockers:
    start = time.perf_counter()
    try:
        completed = subprocess.run(
            native_campaign_command_spec["command"],
            cwd=str(STRATLAKE_ROOT if STRATLAKE_ROOT.exists() else WORKSPACE_ROOT),
            capture_output=True,
            text=True,
            timeout=int(os.environ.get("NOTEBOOK13_CAMPAIGN_TIMEOUT_SECONDS", "3600")),
            check=False,
        )
        runtime_seconds = time.perf_counter() - start
        campaign_execution_result.update(
            {
                "campaign_execution_returncode": completed.returncode,
                "campaign_execution_succeeded": completed.returncode == 0,
                "campaign_execution_runtime_seconds": runtime_seconds,
                "campaign_execution_status": "succeeded" if completed.returncode == 0 else "failed",
                "stdout_tail": tail_text(completed.stdout),
                "stderr_tail": tail_text(completed.stderr),
            }
        )
    except Exception as exc:
        runtime_seconds = time.perf_counter() - start
        campaign_execution_result.update(
            {
                "campaign_execution_runtime_seconds": runtime_seconds,
                "campaign_execution_status": "error",
                "error": repr(exc),
            }
        )
else:
    if campaign_execution_requested:
        campaign_execution_result["campaign_execution_status"] = "blocked"
    else:
        campaign_execution_result["campaign_execution_status"] = "not_requested"

display_markdown("### Campaign execution result")
display_df(pd.DataFrame([campaign_execution_result]))

# Persist command log summary outside Git artifact root.
native_command_log_path = NOTEBOOK13_SUMMARY_ROOT / "native_command_log_summary.json"
write_json(
    native_command_log_path,
    {
        "command_discovery": command_discovery_df.to_dict(orient="records") if not command_discovery_df.empty else [],
        "import_surface_discovery": import_surface_df.to_dict(orient="records") if not import_surface_df.empty else [],
        "entry_points": entry_point_df.to_dict(orient="records") if not entry_point_df.empty else [],
        "native_campaign_command_spec": native_campaign_command_spec,
        "campaign_execution_result": campaign_execution_result,
    },
)
print("Wrote:", native_command_log_path.as_posix())


## 11. Artifact discovery after execution or preview

Artifact discovery runs even when execution is skipped. This lets the notebook inventory any native artifacts already present from a prior manual run while still clearly recording whether Notebook 13 executed the campaign in the current session.


In [ ]:
ARTIFACT_SCAN_ROOTS = [
    NOTEBOOK13_ARTIFACT_ROOT,
    NOTEBOOK13_ARTIFACT_ROOT / "native_campaign",
    STRATLAKE_ROOT / "artifacts",
    STRATLAKE_ROOT / "runs",
    REPO_ROOT / "artifacts",
]

ARTIFACT_SUFFIXES = {
    ".json",
    ".jsonl",
    ".csv",
    ".parquet",
    ".txt",
    ".log",
    ".md",
    ".html",
    ".yaml",
    ".yml",
}


def classify_artifact_origin(path: Path) -> str:
    normalized = path.as_posix().lower()
    if "notebook_13_native_campaign_execution_and_artifact_generation" in normalized:
        if "/native_campaign/" in normalized or "\\native_campaign\\" in normalized:
            return "native_campaign_artifact_candidate"
        return "notebook13_summary_artifact"
    if "governance" in normalized:
        return "native_campaign_governance_artifact"
    if "evidence" in normalized or "review" in normalized:
        return "native_campaign_evidence_review"
    if "campaign" in normalized:
        return "native_campaign_artifact"
    if "notebook_12" in normalized:
        return "prior_notebook_review_artifact"
    return "candidate_campaign_context"


def classify_artifact_role(path: Path) -> str:
    name = path.name.lower()
    full = path.as_posix().lower()
    if "manifest" in name:
        return "manifest"
    if "registry" in name:
        return "run_registry"
    if "split" in name and "metric" in name:
        return "split_metrics"
    if "metric" in name:
        return "metrics"
    if "governance" in full:
        return "governance"
    if "report" in name or path.suffix.lower() in {".md", ".html"}:
        return "report"
    if "log" in name or path.suffix.lower() == ".log":
        return "log"
    if "handoff" in name:
        return "handoff"
    return "artifact"


def discover_artifacts(scan_roots: list[Path], max_files: int = 5000) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    seen: set[str] = set()
    for root in scan_roots:
        if not root.exists():
            rows.append(
                {
                    "scan_root": root.as_posix(),
                    "artifact_path": "",
                    "exists": False,
                    "origin": "missing_scan_root",
                    "role": "missing",
                    "suffix": "",
                    "size_bytes": 0,
                    "modified_utc": "",
                }
            )
            continue
        for path in root.rglob("*"):
            if len(rows) >= max_files:
                break
            if not path.is_file():
                continue
            if path.suffix.lower() not in ARTIFACT_SUFFIXES:
                continue
            resolved = path.resolve().as_posix()
            if resolved in seen:
                continue
            seen.add(resolved)
            stat = path.stat()
            rows.append(
                {
                    "scan_root": root.resolve().as_posix(),
                    "artifact_path": resolved,
                    "exists": True,
                    "origin": classify_artifact_origin(path),
                    "role": classify_artifact_role(path),
                    "suffix": path.suffix.lower(),
                    "size_bytes": stat.st_size,
                    "modified_utc": datetime.fromtimestamp(stat.st_mtime, tz=timezone.utc).isoformat(),
                }
            )
    return pd.DataFrame(rows)


campaign_artifact_inventory_df = discover_artifacts(ARTIFACT_SCAN_ROOTS)

if not campaign_artifact_inventory_df.empty:
    native_campaign_artifact_df = campaign_artifact_inventory_df[
        campaign_artifact_inventory_df["origin"].astype(str).str.startswith("native_campaign")
        | (campaign_artifact_inventory_df["origin"] == "native_campaign_artifact_candidate")
    ].copy()
else:
    native_campaign_artifact_df = pd.DataFrame()

artifact_inventory_path = NOTEBOOK13_INVENTORY_ROOT / "campaign_artifact_inventory.csv"
write_dataframe_csv(artifact_inventory_path, campaign_artifact_inventory_df)

display_markdown("### Campaign artifact inventory")
display_df(campaign_artifact_inventory_df)

display_markdown("### Native campaign artifact candidates")
display_df(native_campaign_artifact_df)

print("Wrote:", artifact_inventory_path.as_posix())


## 12. Campaign artifact inventory summary

This section preserves Notebook 12-compatible handoff fields and separates native campaign artifacts from Notebook 13 summary artifacts.


In [ ]:
def count_role(df: pd.DataFrame, role: str) -> int:
    if df.empty or "role" not in df.columns:
        return 0
    return int((df["role"] == role).sum())


def count_origin_contains(df: pd.DataFrame, token: str) -> int:
    if df.empty or "origin" not in df.columns:
        return 0
    return int(df["origin"].astype(str).str.contains(token, case=False, na=False).sum())


campaign_artifact_rows = int(len(campaign_artifact_inventory_df))
native_campaign_artifact_rows = int(len(native_campaign_artifact_df))
native_campaign_marker_rows = int(
    count_origin_contains(campaign_artifact_inventory_df, "native_campaign")
    + count_role(campaign_artifact_inventory_df, "manifest")
    + count_role(campaign_artifact_inventory_df, "run_registry")
)

candidate_campaign_context_rows = int(
    (campaign_artifact_inventory_df.get("origin", pd.Series(dtype=str)) == "candidate_campaign_context").sum()
    if not campaign_artifact_inventory_df.empty
    else 0
)

campaign_context_loaded = bool(native_campaign_artifact_rows > 0 or candidate_campaign_context_rows > 0)
campaign_review_rows = int(
    count_origin_contains(campaign_artifact_inventory_df, "evidence")
    + count_origin_contains(campaign_artifact_inventory_df, "review")
)

artifact_summary = {
    "campaign_artifact_rows": campaign_artifact_rows,
    "native_campaign_artifact_rows": native_campaign_artifact_rows,
    "native_campaign_marker_rows": native_campaign_marker_rows,
    "candidate_campaign_context_rows": candidate_campaign_context_rows,
    "campaign_context_loaded": campaign_context_loaded,
    "campaign_review_rows": campaign_review_rows,
    "manifest_rows": count_role(campaign_artifact_inventory_df, "manifest"),
    "run_registry_rows": count_role(campaign_artifact_inventory_df, "run_registry"),
    "metrics_rows": count_role(campaign_artifact_inventory_df, "metrics"),
    "split_metrics_rows": count_role(campaign_artifact_inventory_df, "split_metrics"),
    "report_rows": count_role(campaign_artifact_inventory_df, "report"),
    "governance_rows": count_role(campaign_artifact_inventory_df, "governance"),
    "log_rows": count_role(campaign_artifact_inventory_df, "log"),
    "created_at_utc": utc_now_iso(),
}

display_df(pd.DataFrame([artifact_summary]))


## 13. Lightweight metrics/report loading

This section performs shallow loading only. It does not recompute metrics and does not infer promotion readiness from filenames alone.


In [ ]:
def read_json_safely(path: Path) -> Any:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        return {"_read_error": repr(exc), "_path": path.as_posix()}


def read_csv_safely(path: Path, max_rows: int = 1000) -> pd.DataFrame:
    try:
        return pd.read_csv(path, nrows=max_rows)
    except Exception as exc:
        return pd.DataFrame([{"_read_error": repr(exc), "_path": path.as_posix()}])


def load_artifact_previews(inventory_df: pd.DataFrame, max_files_per_role: int = 5) -> dict[str, Any]:
    previews: dict[str, Any] = {}
    if inventory_df.empty:
        return previews
    for role in ["manifest", "run_registry", "metrics", "split_metrics", "governance", "report", "handoff"]:
        role_df = inventory_df[inventory_df["role"] == role].head(max_files_per_role)
        previews[role] = []
        for _, row in role_df.iterrows():
            path = Path(row["artifact_path"])
            item: dict[str, Any] = {
                "artifact_path": path.as_posix(),
                "suffix": path.suffix.lower(),
                "size_bytes": row.get("size_bytes", 0),
            }
            if path.suffix.lower() == ".json":
                content = read_json_safely(path)
                if isinstance(content, dict):
                    item["top_level_keys"] = sorted(list(content.keys()))[:50]
                elif isinstance(content, list):
                    item["list_length"] = len(content)
            elif path.suffix.lower() == ".csv":
                df = read_csv_safely(path, max_rows=50)
                item["columns"] = list(df.columns)
                item["row_preview_count"] = len(df)
            elif path.suffix.lower() in {".md", ".txt", ".log"}:
                try:
                    item["text_tail"] = tail_text(path.read_text(encoding="utf-8", errors="replace"), max_chars=1000)
                except Exception as exc:
                    item["read_error"] = repr(exc)
            previews[role].append(item)
    return previews


artifact_previews = load_artifact_previews(campaign_artifact_inventory_df)

artifact_preview_path = NOTEBOOK13_SUMMARY_ROOT / "campaign_artifact_previews.json"
write_json(artifact_preview_path, artifact_previews)

display_markdown("### Artifact preview role counts")
display_df(pd.DataFrame([{"role": k, "preview_items": len(v)} for k, v in artifact_previews.items()]))

print("Wrote:", artifact_preview_path.as_posix())


## 14. Optional report, evidence, and governance command execution

Optional commands run only when execution is enabled and the native command exists. Unavailable commands are recorded as caveats, not failures.

The notebook does not fabricate reports or governance outputs.


In [ ]:
OPTIONAL_NATIVE_COMMANDS = [
    {
        "surface": "campaign_report",
        "command_name": "stratlake-build-campaign-report",
        "enabled": RUN_OPTIONAL_REPORT_COMMANDS,
    },
    {
        "surface": "evidence_review",
        "command_name": "stratlake-build-evidence-review",
        "enabled": RUN_OPTIONAL_REPORT_COMMANDS,
    },
    {
        "surface": "promotion_governance",
        "command_name": "stratlake-run-promotion-governance-report",
        "enabled": RUN_OPTIONAL_GOVERNANCE_COMMANDS,
    },
]


def build_optional_command(command_name: str, surface: str) -> dict[str, Any]:
    flags = flags_for_command(command_name)
    command = [command_name]
    caveats: list[str] = []

    if not command_available(command_name):
        return {
            "surface": surface,
            "command_name": command_name,
            "command": [],
            "command_string": "",
            "build_succeeded": False,
            "caveats": ["command unavailable"],
            "flags_detected": sorted(flags),
        }

    artifact_added, artifact_flag = add_first_supported_flag(
        command,
        flags,
        ["--artifact-root", "--artifacts-root", "--input-root", "--campaign-root", "--run-root"],
        NOTEBOOK13_ARTIFACT_ROOT / "native_campaign",
    )
    if not artifact_added:
        caveats.append("no artifact/input root flag detected")

    output_added, output_flag = add_first_supported_flag(
        command,
        flags,
        ["--output-root", "--report-root", "--governance-root"],
        NOTEBOOK13_ARTIFACT_ROOT / surface,
    )
    if not output_added:
        caveats.append("no output/report root flag detected")

    if selected_campaign_config_path is not None and selected_campaign_config_path.exists():
        config_added, config_flag = add_first_supported_flag(
            command,
            flags,
            ["--config", "--campaign-config", "--config-path"],
            selected_campaign_config_path,
        )
        if not config_added:
            config_flag = ""
            caveats.append("no config flag detected")
    else:
        config_flag = ""
        caveats.append("campaign config unavailable")

    return {
        "surface": surface,
        "command_name": command_name,
        "command": command,
        "command_string": " ".join(shlex.quote(part) for part in command),
        "build_succeeded": True,
        "artifact_flag": artifact_flag,
        "output_flag": output_flag,
        "config_flag": config_flag,
        "caveats": caveats,
        "flags_detected": sorted(flags),
    }


optional_command_results: list[dict[str, Any]] = []

for spec in OPTIONAL_NATIVE_COMMANDS:
    command_spec = build_optional_command(spec["command_name"], spec["surface"])
    result_record = {
        **command_spec,
        "requested": bool(spec["enabled"]),
        "execution_enabled": bool(spec["enabled"] and NOTEBOOK13_ALLOW_NATIVE_EXECUTION),
        "returncode": None,
        "succeeded": False,
        "status": "not_requested" if not spec["enabled"] else "blocked",
        "stdout_tail": "",
        "stderr_tail": "",
        "error": "",
    }
    if spec["enabled"] and NOTEBOOK13_ALLOW_NATIVE_EXECUTION and command_spec.get("build_succeeded") and command_spec.get("command"):
        try:
            completed = subprocess.run(
                command_spec["command"],
                cwd=str(STRATLAKE_ROOT if STRATLAKE_ROOT.exists() else WORKSPACE_ROOT),
                capture_output=True,
                text=True,
                timeout=int(os.environ.get("NOTEBOOK13_OPTIONAL_COMMAND_TIMEOUT_SECONDS", "1200")),
                check=False,
            )
            result_record.update(
                {
                    "returncode": completed.returncode,
                    "succeeded": completed.returncode == 0,
                    "status": "succeeded" if completed.returncode == 0 else "failed",
                    "stdout_tail": tail_text(completed.stdout),
                    "stderr_tail": tail_text(completed.stderr),
                }
            )
        except Exception as exc:
            result_record.update({"status": "error", "error": repr(exc)})
    elif spec["enabled"]:
        result_record["status"] = "blocked"

    optional_command_results.append(result_record)

optional_command_df = pd.DataFrame(optional_command_results)

optional_command_path = NOTEBOOK13_SUMMARY_ROOT / "optional_report_governance_command_summary.json"
write_json(optional_command_path, optional_command_results)

display_df(optional_command_df)
print("Wrote:", optional_command_path.as_posix())


## 15. Archive checkpoint

Archive checkpointing is optional and native-command-only. It requires:

- archive checkpoint profile,
- `NOTEBOOK13_ALLOW_ARCHIVE_CHECKPOINT=true`,
- native archive command availability,
- explicit archive root.

Google Drive is treated as session persistence, not active working storage.


In [ ]:
NOTEBOOK13_DRIVE_ARCHIVE_ROOT = Path(
    os.environ.get(
        "NOTEBOOK13_DRIVE_ARCHIVE_ROOT",
        "/content/drive/MyDrive/stratlake-colab/session_archives" if IN_COLAB else str(NOTEBOOK13_ARTIFACT_ROOT / "session_archives"),
    )
).expanduser()

archive_checkpoint_requested = bool(RUN_ARCHIVE_CHECKPOINT)
archive_checkpoint_enabled = bool(RUN_ARCHIVE_CHECKPOINT and NOTEBOOK13_ALLOW_ARCHIVE_CHECKPOINT)

archive_command = [
    "stratlake-session-archive-bootstrap",
    "--root",
    str(STRATLAKE_ROOT),
    "--archive-id",
    os.environ.get("NOTEBOOK13_ARCHIVE_ID", "notebook13-native-campaign-execution"),
    "--archive-collision-policy",
    os.environ.get("NOTEBOOK13_ARCHIVE_COLLISION_POLICY", "overwrite_allowed"),
    "--drive-root",
    str(NOTEBOOK13_DRIVE_ARCHIVE_ROOT),
    "--copy-policy",
    os.environ.get("NOTEBOOK13_ARCHIVE_COPY_POLICY", "overwrite_allowed"),
    "--include-features",
    "--include-artifacts",
    "--include-configs",
    "--validate-after-copy",
    "--inspect-after-copy",
]

archive_checkpoint_result = {
    "archive_checkpoint_requested": archive_checkpoint_requested,
    "archive_checkpoint_enabled": archive_checkpoint_enabled,
    "archive_command": " ".join(shlex.quote(part) for part in archive_command),
    "archive_returncode": None,
    "archive_succeeded": False,
    "archive_status": "not_requested" if not archive_checkpoint_requested else "blocked",
    "stdout_tail": "",
    "stderr_tail": "",
    "error": "",
    "caveats": [],
}

archive_blockers: list[str] = []
if archive_checkpoint_requested and not NOTEBOOK13_ALLOW_ARCHIVE_CHECKPOINT:
    archive_blockers.append("NOTEBOOK13_ALLOW_ARCHIVE_CHECKPOINT is not true")
if not command_available(archive_command[0]):
    archive_blockers.append("native archive checkpoint command unavailable")
if not campaign_execution_result.get("campaign_execution_succeeded", False):
    archive_blockers.append("campaign execution did not succeed in this session")

archive_checkpoint_result["caveats"] = archive_blockers

if archive_checkpoint_enabled and not archive_blockers:
    ensure_dir(NOTEBOOK13_DRIVE_ARCHIVE_ROOT)
    try:
        completed = subprocess.run(
            archive_command,
            cwd=str(STRATLAKE_ROOT if STRATLAKE_ROOT.exists() else WORKSPACE_ROOT),
            capture_output=True,
            text=True,
            timeout=int(os.environ.get("NOTEBOOK13_ARCHIVE_TIMEOUT_SECONDS", "1800")),
            check=False,
        )
        archive_checkpoint_result.update(
            {
                "archive_returncode": completed.returncode,
                "archive_succeeded": completed.returncode == 0,
                "archive_status": "succeeded" if completed.returncode == 0 else "failed",
                "stdout_tail": tail_text(completed.stdout),
                "stderr_tail": tail_text(completed.stderr),
            }
        )
    except Exception as exc:
        archive_checkpoint_result.update({"archive_status": "error", "error": repr(exc)})

archive_checkpoint_path = NOTEBOOK13_SUMMARY_ROOT / "archive_checkpoint_summary.json"
write_json(archive_checkpoint_path, archive_checkpoint_result)

display_df(pd.DataFrame([archive_checkpoint_result]))
print("Wrote:", archive_checkpoint_path.as_posix())


## 16. Campaign caveat register

Caveats are evidence-preserving. They prevent Notebook 13 from overstating execution, governance, or promotion readiness.


In [ ]:
caveat_rows: list[dict[str, Any]] = []

def add_caveat(surface: str, caveat: str, severity: str = "info") -> None:
    if caveat:
        caveat_rows.append(
            {
                "surface": surface,
                "severity": severity,
                "caveat": caveat,
                "created_at_utc": utc_now_iso(),
            }
        )

for blocker in preflight_status.get("preflight_hard_blockers", []):
    add_caveat("campaign_preflight", blocker, "error")

for caveat in preflight_status.get("preflight_caveats", []):
    add_caveat("campaign_preflight", caveat, "warning")

for caveat in archive_restore_result.get("caveats", []):
    add_caveat("archive_restore", caveat, "info")

for caveat in campaign_execution_result.get("caveats", []):
    add_caveat("campaign_execution", caveat, "warning")

for row in optional_command_results:
    for caveat in row.get("caveats", []):
        add_caveat(row.get("surface", "optional_command"), caveat, "info")

for caveat in archive_checkpoint_result.get("caveats", []):
    add_caveat("archive_checkpoint", caveat, "info")

if not campaign_execution_result.get("campaign_execution_succeeded", False):
    add_caveat(
        "promotion_boundary",
        "promotion-grade readiness is not claimed because native campaign execution did not succeed in this session",
        "warning",
    )

if artifact_summary.get("governance_rows", 0) == 0 and not any(row.get("surface") == "promotion_governance" and row.get("succeeded") for row in optional_command_results):
    add_caveat(
        "promotion_boundary",
        "native promotion governance artifacts were not detected or generated",
        "warning",
    )

if campaign_config_is_notebook_generated:
    add_caveat(
        "campaign_config",
        "selected campaign config is notebook-generated and must not be treated as a native StratLake template",
        "warning",
    )

if preflight_status.get("universe_config_is_notebook_generated", False):
    add_caveat(
        "universe_config",
        "selected universe config is notebook-generated and must not be treated as native StratLake execution evidence",
        "warning",
    )

if not preflight_status.get("native_execution_input_ready", False):
    add_caveat(
        "execution_boundary",
        "native execution input readiness is false until campaign config, universe config, and feature input root are user-reviewed/native via NOTEBOOK13_MARK_INPUTS_USER_REVIEWED or specific reviewed flags",
        "warning",
    )

campaign_caveat_register_df = pd.DataFrame(caveat_rows)

campaign_caveat_register_path = NOTEBOOK13_INVENTORY_ROOT / "campaign_caveat_register.csv"
write_dataframe_csv(campaign_caveat_register_path, campaign_caveat_register_df)

display_df(campaign_caveat_register_df)
print("Wrote:", campaign_caveat_register_path.as_posix())


## 17. Notebook 12-compatible handoff summary

The handoff records what Notebook 13 actually did and what artifacts are reviewable by Notebook 12-style evidence review.


In [ ]:

selected_stratlake_init_rows = session_init_df[
    (session_init_df.get("surface", pd.Series(dtype=str)).astype(str).str.startswith("stratlake"))
    & (session_init_df.get("selected", pd.Series(dtype=bool)).astype(bool))
] if "session_init_df" in globals() and not session_init_df.empty else pd.DataFrame()
selected_stratlake_init_surface = "" if selected_stratlake_init_rows.empty else str(selected_stratlake_init_rows.iloc[0].get("surface", ""))
selected_stratlake_init_command = "" if selected_stratlake_init_rows.empty else str(selected_stratlake_init_rows.iloc[0].get("command", ""))
selected_stratlake_init_succeeded = False if selected_stratlake_init_rows.empty else bool(selected_stratlake_init_rows.iloc[0].get("succeeded", False))

campaign_report_available = bool(
    artifact_summary.get("report_rows", 0) > 0
    or any(row.get("surface") == "campaign_report" and row.get("succeeded") for row in optional_command_results)
)

evidence_review_available = bool(
    artifact_summary.get("campaign_review_rows", 0) > 0
    or any(row.get("surface") == "evidence_review" and row.get("succeeded") for row in optional_command_results)
)

promotion_governance_available = bool(
    artifact_summary.get("governance_rows", 0) > 0
    or any(row.get("surface") == "promotion_governance" and row.get("succeeded") for row in optional_command_results)
)

if campaign_execution_result.get("campaign_execution_succeeded") and native_campaign_artifact_rows > 0:
    notebook13_handoff_status = "notebook_13_native_campaign_execution_smoke_passed_with_artifacts"
elif campaign_execution_result.get("campaign_execution_succeeded"):
    notebook13_handoff_status = "notebook_13_campaign_execution_completed_artifacts_not_detected"
elif campaign_execution_requested:
    notebook13_handoff_status = "notebook_13_native_campaign_execution_blocked_with_caveats"
else:
    notebook13_handoff_status = "notebook_13_native_campaign_execution_import_ready_runtime_execution_manual"

remaining_caveats = campaign_caveat_register_df["caveat"].tolist() if not campaign_caveat_register_df.empty else []

notebook13_handoff_summary = {
    "notebook13_handoff_status": notebook13_handoff_status,
    "selected_stratlake_init_surface": selected_stratlake_init_surface,
    "selected_stratlake_init_command": selected_stratlake_init_command,
    "selected_stratlake_init_succeeded": selected_stratlake_init_succeeded,
    "archive_restore_requested": archive_restore_result.get("archive_restore_requested", False),
    "archive_restore_enabled": archive_restore_result.get("archive_restore_enabled", False),
    "archive_restore_succeeded": archive_restore_result.get("archive_restore_succeeded", False),
    "archive_restore_status": archive_restore_result.get("archive_restore_status", ""),
    "restore_archive_id": archive_restore_result.get("restore_archive_id", ""),
    "campaign_execution_requested": campaign_execution_requested,
    "campaign_execution_enabled": campaign_execution_enabled,
    "campaign_execution_succeeded": bool(campaign_execution_result.get("campaign_execution_succeeded", False)),
    "campaign_execution_returncode": campaign_execution_result.get("campaign_execution_returncode"),
    "native_campaign_artifact_rows": native_campaign_artifact_rows,
    "campaign_artifact_rows": campaign_artifact_rows,
    "campaign_context_loaded": campaign_context_loaded,
    "campaign_report_available": campaign_report_available,
    "evidence_review_available": evidence_review_available,
    "promotion_governance_available": promotion_governance_available,
    "campaign_config_source": campaign_config_source,
    "campaign_config_path": campaign_config_status["campaign_config_path"],
    "campaign_config_is_native_template": campaign_config_is_native_template,
    "campaign_config_is_notebook_generated": campaign_config_is_notebook_generated,
    "campaign_config_is_user_reviewed": campaign_config_is_user_reviewed,
    "campaign_config_execution_ready": campaign_config_status.get("campaign_config_execution_ready", False),
    "selected_feature_input_path": preflight_status.get("selected_feature_input_path", ""),
    "feature_file_count": preflight_status.get("feature_file_count", 0),
    "feature_files_ready_for_execution": preflight_status.get("feature_files_ready_for_execution", False),
    "feature_input_source": preflight_status.get("feature_input_source", ""),
    "feature_input_is_user_reviewed": preflight_status.get("feature_input_is_user_reviewed", False),
    "selected_universe_config_path": preflight_status.get("selected_universe_config_path", ""),
    "universe_config_source": preflight_status.get("universe_config_source", ""),
    "universe_config_is_notebook_generated": preflight_status.get("universe_config_is_notebook_generated", False),
    "universe_config_is_user_reviewed": preflight_status.get("universe_config_is_user_reviewed", False),
    "native_execution_input_ready": preflight_status.get("native_execution_input_ready", False),
    "mark_inputs_user_reviewed": NOTEBOOK13_MARK_INPUTS_USER_REVIEWED,
    "recommended_next_notebook": "Notebook 14 — Campaign Evidence Review from Notebook 13 Native Artifacts",
    "remaining_caveats": remaining_caveats,
    "promotion_grade_claim_made": False,
    "production_readiness_claim_made": False,
    "statistical_significance_claim_made": False,
    "created_at_utc": utc_now_iso(),
}

campaign_execution_summary_path = NOTEBOOK13_SUMMARY_ROOT / "campaign_execution_summary.json"
campaign_execution_handoff_path = NOTEBOOK13_SUMMARY_ROOT / "campaign_execution_handoff.json"

write_json(
    campaign_execution_summary_path,
    {
        "profile_status": profile_status,
        "workspace_status": workspace_status,
        "campaign_config_status": campaign_config_status,
        "preflight_status": preflight_status,
        "archive_restore_result": archive_restore_result,
        "native_campaign_command_spec": native_campaign_command_spec,
        "campaign_execution_result": campaign_execution_result,
        "artifact_summary": artifact_summary,
        "optional_command_results": optional_command_results,
        "archive_checkpoint_result": archive_checkpoint_result,
        "handoff": notebook13_handoff_summary,
    },
)
write_json(campaign_execution_handoff_path, notebook13_handoff_summary)

display_markdown("### Notebook 13 handoff summary")
display_df(pd.DataFrame([notebook13_handoff_summary]))

print("Wrote:", campaign_execution_summary_path.as_posix())
print("Wrote:", campaign_execution_handoff_path.as_posix())


## 18. Source-safe import checklist

Before committing Notebook 13 to `fintech-stratlake-notebook-workflows`, ensure:

```bash
python scripts/check_notebooks_no_outputs.py notebooks
python scripts/validate_repo_cleanliness.py .
python scripts/scan_for_secret_patterns.py .
pytest
```

Optional focused validation:

```bash
python -m pytest tests/test_notebook_13_source_contracts.py -q
python -m pytest tests/test_notebook_13_execution_guardrails.py -q
python -m pytest tests/test_notebook_13_artifact_handoff_contracts.py -q
```

Expected committed notebook properties:

- output-free,
- execution-count-null,
- cell-ID-clean if repo convention requires it,
- metadata-minimized,
- secret-free,
- artifact-free,
- safe default profile,
- execution requires explicit opt-in,
- generated artifact paths point outside committed source.


## 19. Proposed static tests and documentation

Suggested documentation:

```text
docs/notebook_13_import_audit.md
docs/notebook_13_command_surface_classification.md
docs/notebook_13_execution_surface_classification.md
docs/milestone_16_merge_readiness.md
```

Suggested source/static tests:

```text
tests/test_notebook_13_source_contracts.py
tests/test_notebook_13_execution_guardrails.py
tests/test_notebook_13_artifact_handoff_contracts.py
```

Coverage should include:

- notebook path exists,
- notebook JSON valid,
- no outputs,
- execution counts null,
- required profiles present,
- default profile is safe,
- native execution requires explicit opt-in,
- non-claim flags exist,
- native command names present,
- generated artifact paths are outside Git,
- campaign config source fields exist,
- native artifact origin fields exist,
- Notebook 12-compatible handoff fields exist,
- promotion-grade claims are not made by default.


## 20. Completion stances

Source-safe import planning:

```text
notebook_13_native_campaign_execution_source_safe_import_planned
```

After staged import:

```text
notebook_13_native_campaign_execution_staged_source_safe
```

After guarded runtime smoke, if execution succeeds:

```text
notebook_13_native_campaign_execution_smoke_passed_with_artifacts
```

If execution is blocked:

```text
notebook_13_native_campaign_execution_blocked_with_caveats
```

If full campaign artifacts are produced but promotion evidence remains incomplete:

```text
notebook_13_campaign_execution_completed_artifacts_reviewable_promotion_incomplete
```

Preferred conservative first-import stance:

```text
notebook_13_native_campaign_execution_import_ready_runtime_execution_manual
```
